# Gujarati NLP Mini Project

### Step 1: Preprocessing

In [3]:
%pip install -q indic-nlp-library tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import re
import sys
import unicodedata
from collections import Counter
from functools import lru_cache

import pandas as pd
from tqdm import tqdm
from indicnlp.tokenize import indic_tokenize
from indicnlp.normalize.indic_normalize import IndicNormalizerFactory

tqdm.pandas()
pd.set_option('display.max_colwidth', 200)

In [5]:
DATA_PATH = 'nlp_preprocessed_dataset.csv'

df = pd.read_csv(DATA_PATH, usecols=['id', 'headline', 'text'])
df['text'] = df['text'].fillna('').astype(str)
print('Shape:', df.shape)
df.head()

Shape: (50000, 3)


,id,headline,text
0,n338153624,બાંટવા ગામે રવિવારે જુનાગઢના નામાંકિત ડોકટરો દ્વારા સર્વરોગ નિદાન કેમ્પ યોજાશે,"બાંટવા, તા. 3બાંટવા ગામે આગામી તા.05/12/2021 રવિવારના રોજ સમય સવારે 9:00 થી 1:00 શ્રી લાડી લોહાણા સિંધી પંચાયતની વાડી તળાવ રોડ ગલી નં.7માં લાડી લોહાણા સિંધી પંચાયત બાંટવા દ્વારા આયોજિત લાડી લોહાણ..."
1,n407137262,ગાંધીનગરમાં વરસાદી માહોલ વચ્ચે મચ્છરજન્ય રોગચાળો વકર્યો,ગાંધીનગર જિલ્લામાં વરસાદી માહોલની વચ્ચે મચ્છરજન્ય રોગચાળો ધીમે ધીમે માથું ઊંચકી રહ્યો છે. સતત રહેતાં વાદળછાયા વાતાવરણ તેમજ ઠેર ઠેર વરસાદી પાણી ભરાઈ રહેવાના કારણે સાદો તાવ મેલેરિયા અને ડેન્ગ્યુ જે...
2,n394530274,WWEની મહિલા સુપરસ્ટાર રેસલરે લીધી નિવૃત્તિ,ભારત સહિત આખી દુનિયામાં WWE જોનારનો મોટો વર્ગ છે. પરંતુ હાલ WWEના ચાહકો માટે નિરાશાજનક ખબર સામે આવ્યા છે. બે વખતની WWE ચેમ્પિયન પેજે નિવૃતીની જાહેરાત કરી છે. પેજ 7 જુલાઈએ કરાર સમાપ્ત થશે ત્યારે W...
3,n388799324,"હવે આસામમાં બુલડોઝરની એન્ટ્રી, પોલીસ મથકને આગ લગાવનારા પાંચના ઘર તોડી પડાયા","નવી દિલ્હી,તા. 22 મે 2022 હિંસા પર ઉતરી આવતા લોકોના ઘર પર બુલડોઝર ચલાવવાની નવી પ્રથાની યુપી અને એમપી બાદ હવે આસામમાં પણ એન્ટ્રી થઈ છે.આસામના નગાંવ જિલ્લામાં પોલીસે માછલીના વેપારી સફીકુલ ઈસ્લામને અ..."
4,n404948806,16 વર્ષીય બ્રિટિશ પાયલટનું પ્લેન વરસાદમાં ઈંધણ ભરવા માટે અમદાવાદ એરપોર્ટ પર ઉતર્યું,"દુનિયામાં આવી અનેક ઘટનાઓ છે જે ગિનીસ બુક ઓફ વર્લ્ડ રેકોર્ડમાં નોંધાયેલી છે. પરંતુ આ વખતે એક ઘટના સામે આવી છે જે અલગ છે. માત્ર 16 વર્ષની ઉંમરે, બ્રિટીશ-બેલ્જિયન પાઇલટ મેક રધરફોર્ડ પ્લેનમાં એકલા પૃ..."


In [6]:
# Build translation tables once, covering every script:
#   Nd/No -> digits & numerals (0-9, ૦-૯, ½, ², ...)
#   P*/S* -> punctuation & symbols (. , ! ? । ૰ ₹ “ ” emojis, ...)
DIGIT_TABLE, PUNCT_TABLE = {}, {}
for cp in range(sys.maxunicode + 1):
    cat = unicodedata.category(chr(cp))
    if cat in ('Nd', 'No'):
        DIGIT_TABLE[cp] = ' '
    elif cat[0] in ('P', 'S'):
        PUNCT_TABLE[cp] = ' '

ZERO_WIDTH_RE = re.compile('[\u200b-\u200d\u2060\ufeff\u00ad]')
WHITESPACE_RE = re.compile(r'\s+')


def remove_digits(text):
    return text.translate(DIGIT_TABLE)


def remove_punctuation(text):
    return text.translate(PUNCT_TABLE)


def to_lowercase(text):
    # Gujarati script has no case; this only affects embedded Latin text (e.g. "IPL", "Covid").
    return text.lower()


def strip_extra_spaces(text):
    text = ZERO_WIDTH_RE.sub('', text)
    return WHITESPACE_RE.sub(' ', text).strip()


def clean_text(text):
    text = unicodedata.normalize('NFC', text)
    text = remove_digits(text)
    text = remove_punctuation(text)
    text = to_lowercase(text)
    return strip_extra_spaces(text)

In [7]:
# Why not a \w-based regex? It strips Gujarati matras.
w = 'બાંટવા ગામે રવિવારે, તા.05/12'
print('regex [^\\w\\s] :', re.sub(r'[^\w\s]', '', w))
print('clean_text()    :', clean_text(w))

regex [^\w\s] : બટવ ગમ રવવર ત0512
clean_text()    : બાંટવા ગામે રવિવારે તા


In [8]:
# Step-by-step on one article
s = unicodedata.normalize('NFC', df.loc[0, 'text'][:300])
print('RAW               :', s, '\n')
s = remove_digits(s)
print('Digits removed    :', s, '\n')
s = remove_punctuation(s)
print('Punct removed     :', s, '\n')
s = to_lowercase(s)
print('Lowercased        :', s, '\n')
s = strip_extra_spaces(s)
print('Spaces stripped   :', s)

RAW               :  બાંટવા, તા. 3બાંટવા ગામે આગામી તા.05/12/2021 રવિવારના રોજ સમય સવારે 9:00 થી 1:00 શ્રી લાડી લોહાણા સિંધી પંચાયતની વાડી તળાવ રોડ ગલી નં.7માં લાડી લોહાણા સિંધી પંચાયત બાંટવા દ્વારા આયોજિત લાડી લોહાણા સિંધી નવયુવક મંડળ બાંટવા તેમજ શ્રી લાડી લોહાણા મહિલા મંડળ બાંટવા સહયોગથી જૂનાગઢના નામાંકીત ડોક્ટરો સર્ 

Digits removed    :  બાંટવા, તા.  બાંટવા ગામે આગામી તા.  /  /     રવિવારના રોજ સમય સવારે  :   થી  :   શ્રી લાડી લોહાણા સિંધી પંચાયતની વાડી તળાવ રોડ ગલી નં. માં લાડી લોહાણા સિંધી પંચાયત બાંટવા દ્વારા આયોજિત લાડી લોહાણા સિંધી નવયુવક મંડળ બાંટવા તેમજ શ્રી લાડી લોહાણા મહિલા મંડળ બાંટવા સહયોગથી જૂનાગઢના નામાંકીત ડોક્ટરો સર્ 

Punct removed     :  બાંટવા  તા   બાંટવા ગામે આગામી તા            રવિવારના રોજ સમય સવારે      થી      શ્રી લાડી લોહાણા સિંધી પંચાયતની વાડી તળાવ રોડ ગલી નં  માં લાડી લોહાણા સિંધી પંચાયત બાંટવા દ્વારા આયોજિત લાડી લોહાણા સિંધી નવયુવક મંડળ બાંટવા તેમજ શ્રી લાડી લોહાણા મહિલા મંડળ બાંટવા સહયોગથી જૂનાગઢના નામાંકીત ડોક્ટરો સર્ 

Lowercased        :  બાંટવા  તા

In [9]:
df['clean_text'] = df['text'].progress_apply(clean_text)

empty = (df['clean_text'].str.len() == 0).sum()
df = df[df['clean_text'].str.len() > 0].reset_index(drop=True)
print(f'Dropped {empty} articles that were empty after cleaning; {len(df)} remain.')
df[['text', 'clean_text']].head(3)

100%|██████████| 50000/50000 [01:12<00:00, 689.78it/s]


Dropped 0 articles that were empty after cleaning; 50000 remain.


,text,clean_text
0,"બાંટવા, તા. 3બાંટવા ગામે આગામી તા.05/12/2021 રવિવારના રોજ સમય સવારે 9:00 થી 1:00 શ્રી લાડી લોહાણા સિંધી પંચાયતની વાડી તળાવ રોડ ગલી નં.7માં લાડી લોહાણા સિંધી પંચાયત બાંટવા દ્વારા આયોજિત લાડી લોહાણ...",બાંટવા તા બાંટવા ગામે આગામી તા રવિવારના રોજ સમય સવારે થી શ્રી લાડી લોહાણા સિંધી પંચાયતની વાડી તળાવ રોડ ગલી નં માં લાડી લોહાણા સિંધી પંચાયત બાંટવા દ્વારા આયોજિત લાડી લોહાણા સિંધી નવયુવક મંડળ બાંટવા...
1,ગાંધીનગર જિલ્લામાં વરસાદી માહોલની વચ્ચે મચ્છરજન્ય રોગચાળો ધીમે ધીમે માથું ઊંચકી રહ્યો છે. સતત રહેતાં વાદળછાયા વાતાવરણ તેમજ ઠેર ઠેર વરસાદી પાણી ભરાઈ રહેવાના કારણે સાદો તાવ મેલેરિયા અને ડેન્ગ્યુ જે...,ગાંધીનગર જિલ્લામાં વરસાદી માહોલની વચ્ચે મચ્છરજન્ય રોગચાળો ધીમે ધીમે માથું ઊંચકી રહ્યો છે સતત રહેતાં વાદળછાયા વાતાવરણ તેમજ ઠેર ઠેર વરસાદી પાણી ભરાઈ રહેવાના કારણે સાદો તાવ મેલેરિયા અને ડેન્ગ્યુ જેવા...
2,ભારત સહિત આખી દુનિયામાં WWE જોનારનો મોટો વર્ગ છે. પરંતુ હાલ WWEના ચાહકો માટે નિરાશાજનક ખબર સામે આવ્યા છે. બે વખતની WWE ચેમ્પિયન પેજે નિવૃતીની જાહેરાત કરી છે. પેજ 7 જુલાઈએ કરાર સમાપ્ત થશે ત્યારે W...,ભારત સહિત આખી દુનિયામાં wwe જોનારનો મોટો વર્ગ છે પરંતુ હાલ wweના ચાહકો માટે નિરાશાજનક ખબર સામે આવ્યા છે બે વખતની wwe ચેમ્પિયન પેજે નિવૃતીની જાહેરાત કરી છે પેજ જુલાઈએ કરાર સમાપ્ત થશે ત્યારે wwe છોડ...


In [10]:
GUJARATI_STOPWORDS = '''
હું મેં મને મારું મારા મારી મારો અમે અમને અમારું અમારા અમારી અમારો
આપણે આપણને આપણું આપણા આપણી આપણો તું તને તારું તારા તારી તારો
તમે તમને તમારું તમારા તમારી તમારો
તે તેણે તેને તેનું તેના તેની તેનો તેમાં તેઓ તેઓએ તેમને તેમનું તેમના તેમની તેમનો તેમણે
આ એ આને એને એનું એના એની એનો આમાં એમાં આના આની આનું
જે જેને જેનું જેના જેની જેનો જેમાં જેણે જેઓ
કોઈ કોણ કોને કંઈ કશું શું બધા બધાં બધું બધી દરેક પોતે પોતાના પોતાની પોતાનું પોતાનો
ના ની નું નો ને થી માં પર પરથી માટે સાથે વિશે વિષે દ્વારા સુધી પાસે પાસેથી તરફ તરફથી
બાદ પછી પહેલા પહેલાં વચ્ચે અંદર બહાર ઉપર નીચે સામે
જેવું જેવા જેવી જેવો જેમ તેમ તેમજ વગેરે
જ પણ તો કે અને અથવા પરંતુ છતાં કારણ કેમકે જો ત્યારે જ્યારે ત્યાં જ્યાં અહીં
હવે હજુ હજી ફક્ત માત્ર ખૂબ બહુ વધુ વધારે ઘણા ઘણી ઘણું એટલે એટલું એવું એવા એવી
કેમ ક્યાં ક્યારે કેવી કેવું કેટલા કેટલું નહીં નહિ નહી નથી હા એક
છે છો છું છીએ હતો હતી હતું હતા હતાં હશે હોય હોવા હોવાથી
થયો થયું થઈ થઇ થયા થયેલ થયેલા થાય થશે થવા થતા થતાં
કર્યો કર્યું કરી કર્યા કરે કરવા કરવામાં કરવાની કરશે કરતા કરતાં કરીને કરાયો કરાયું કરાઈ કરાઇ
રહ્યો રહ્યું રહ્યા રહી રહે રહેલ
આવ્યો આવ્યું આવી આવ્યા આવે આવશે આવેલ આવેલા ગયો ગયું ગઈ ગઇ ગયા
'''


def tokenize(text):
    return indic_tokenize.trivial_tokenize(text, lang='gu')


normalizer = IndicNormalizerFactory().get_normalizer('gu')


@lru_cache(maxsize=None)
def normalize_token(token):
    return normalizer.normalize(token)


STOPWORDS = {normalize_token(unicodedata.normalize('NFC', w)) for w in GUJARATI_STOPWORDS.split()}


def remove_stopwords(tokens):
    return [t for t in tokens if t and t not in STOPWORDS]


print('Stopwords:', len(STOPWORDS))

Stopwords: 228


In [11]:
# Step-by-step on one article
tokens = tokenize(df.loc[0, 'clean_text'])
normalized = [normalize_token(t) for t in tokens]
final = remove_stopwords(normalized)
changed = [f'{a} -> {b}' for a, b in zip(tokens, normalized) if a != b]

print('Tokens            :', len(tokens), tokens[:30], '\n')
print('Normalized        :', changed[:10] or 'no tokens changed in this article', '\n')
print('Stopwords removed :', len(normalized) - len(final), sorted(set(normalized) & STOPWORDS), '\n')
print('Final             :', len(final), final[:30])

Tokens            : 114 ['બાંટવા', 'તા', 'બાંટવા', 'ગામે', 'આગામી', 'તા', 'રવિવારના', 'રોજ', 'સમય', 'સવારે', 'થી', 'શ્રી', 'લાડી', 'લોહાણા', 'સિંધી', 'પંચાયતની', 'વાડી', 'તળાવ', 'રોડ', 'ગલી', 'નં', 'માં', 'લાડી', 'લોહાણા', 'સિંધી', 'પંચાયત', 'બાંટવા', 'દ્વારા', 'આયોજિત', 'લાડી'] 

Normalized        : no tokens changed in this article 

Stopwords removed : 9 ['આ', 'છે', 'તેમજ', 'થી', 'દ્વારા', 'પર', 'માં', 'માટે'] 

Final             : 105 ['બાંટવા', 'તા', 'બાંટવા', 'ગામે', 'આગામી', 'તા', 'રવિવારના', 'રોજ', 'સમય', 'સવારે', 'શ્રી', 'લાડી', 'લોહાણા', 'સિંધી', 'પંચાયતની', 'વાડી', 'તળાવ', 'રોડ', 'ગલી', 'નં', 'લાડી', 'લોહાણા', 'સિંધી', 'પંચાયત', 'બાંટવા', 'આયોજિત', 'લાડી', 'લોહાણા', 'સિંધી', 'નવયુવક']


In [12]:
# Full dataset. Token lists are not kept per document (memory); final tokens are stored space-joined.
raw_vocab, norm_vocab, final_vocab = Counter(), Counter(), Counter()
processed, n_tokens, n_final = [], [], []

for text in tqdm(df['clean_text'], desc='Tokenize / normalize / stopwords'):
    tokens = tokenize(text)
    normalized = [normalize_token(t) for t in tokens]
    final = remove_stopwords(normalized)

    raw_vocab.update(tokens)
    norm_vocab.update(normalized)
    final_vocab.update(final)
    processed.append(' '.join(final))
    n_tokens.append(len(tokens))
    n_final.append(len(final))

df['processed_text'] = processed
df['n_tokens'] = n_tokens
df['n_tokens_final'] = n_final
df[['clean_text', 'processed_text']].head(3)

Tokenize / normalize / stopwords: 100%|██████████| 50000/50000 [00:54<00:00, 916.93it/s] 


,clean_text,processed_text
0,બાંટવા તા બાંટવા ગામે આગામી તા રવિવારના રોજ સમય સવારે થી શ્રી લાડી લોહાણા સિંધી પંચાયતની વાડી તળાવ રોડ ગલી નં માં લાડી લોહાણા સિંધી પંચાયત બાંટવા દ્વારા આયોજિત લાડી લોહાણા સિંધી નવયુવક મંડળ બાંટવા...,બાંટવા તા બાંટવા ગામે આગામી તા રવિવારના રોજ સમય સવારે શ્રી લાડી લોહાણા સિંધી પંચાયતની વાડી તળાવ રોડ ગલી નં લાડી લોહાણા સિંધી પંચાયત બાંટવા આયોજિત લાડી લોહાણા સિંધી નવયુવક મંડળ બાંટવા શ્રી લાડી લોહ...
1,ગાંધીનગર જિલ્લામાં વરસાદી માહોલની વચ્ચે મચ્છરજન્ય રોગચાળો ધીમે ધીમે માથું ઊંચકી રહ્યો છે સતત રહેતાં વાદળછાયા વાતાવરણ તેમજ ઠેર ઠેર વરસાદી પાણી ભરાઈ રહેવાના કારણે સાદો તાવ મેલેરિયા અને ડેન્ગ્યુ જેવા...,ગાંધીનગર જિલ્લામાં વરસાદી માહોલની મચ્છરજન્ય રોગચાળો ધીમે ધીમે માથું ઊંચકી સતત રહેતાં વાદળછાયા વાતાવરણ ઠેર ઠેર વરસાદી પાણી ભરાઈ રહેવાના કારણે સાદો તાવ મેલેરિયા ડેન્ગ્યુ રોગોમાં વધારો ગાંધીનગરમાં છે...
2,ભારત સહિત આખી દુનિયામાં wwe જોનારનો મોટો વર્ગ છે પરંતુ હાલ wweના ચાહકો માટે નિરાશાજનક ખબર સામે આવ્યા છે બે વખતની wwe ચેમ્પિયન પેજે નિવૃતીની જાહેરાત કરી છે પેજ જુલાઈએ કરાર સમાપ્ત થશે ત્યારે wwe છોડ...,ભારત સહિત આખી દુનિયામાં wwe જોનારનો મોટો વર્ગ હાલ wweના ચાહકો નિરાશાજનક ખબર બે વખતની wwe ચેમ્પિયન પેજે નિવૃતીની જાહેરાત પેજ જુલાઈએ કરાર સમાપ્ત wwe છોડી દેશે વર્ષીય પેજે ટ્વિટર માહિતી આપી પેજે ટ્વિ...


In [13]:
total_raw, total_final = sum(raw_vocab.values()), sum(final_vocab.values())

stats = pd.Series({
    'Documents': len(df),
    'Total tokens (after tokenization)': total_raw,
    'Total tokens (after stopword removal)': total_final,
    'Stopword tokens removed (%)': round(100 * (1 - total_final / total_raw), 2),
    'Vocabulary size: raw tokens': len(raw_vocab),
    'Vocabulary size: after normalization': len(norm_vocab),
    'Vocabulary size: after stopword removal': len(final_vocab),
    'Avg tokens / doc (before stopwords)': round(df['n_tokens'].mean(), 1),
    'Avg tokens / doc (after stopwords)': round(df['n_tokens_final'].mean(), 1),
})
stats.to_frame('value')

,value
Documents,50000.00
Total tokens (after tokenization),14629680.00
Total tokens (after stopword removal),10140610.00
Stopword tokens removed (%),30.68
Vocabulary size: raw tokens,418260.00
Vocabulary size: after normalization,418260.00
Vocabulary size: after stopword removal,418032.00
Avg tokens / doc (before stopwords),292.60
Avg tokens / doc (after stopwords),202.80


In [14]:
# Most frequent tokens before vs after stopword removal.
# Review the right-hand list: frequent function words still present are candidates to add to GUJARATI_STOPWORDS.
top_before = pd.DataFrame(norm_vocab.most_common(25), columns=['token', 'count'])
top_after = pd.DataFrame(final_vocab.most_common(25), columns=['token', 'count'])
pd.concat({'Before stopword removal': top_before, 'After stopword removal': top_after}, axis=1)

Before stopword removal         After stopword removal       
                     token   count                  token  count
0                       છે  596172                 કહ્યું  31406
1                      અને  266289                    શકે  30721
2                        આ  204336                  કારણે  25771
3                       કે  160787                    મળી  25048
4                       પણ  140694                      ન  24982
5                     માટે  113192                    આજે  24466
6                       પર  108458                   લોકો  24128
7                      કરી  100952                   રીતે  23694
8                      હતી   93151                  પોલીસ  22114
9                        જ   80433                     બે  21886
10                      એક   79076               જણાવ્યું  19930
11                    સાથે   78810                   જોવા  19467
12                     હતો   68049                   તમામ  19355
13                      તે   66715                    આપી  19336
14                     આવી   64736                દરમિયાન  18902
15                     હતા   62703                    શરૂ  17395
16                 કરવામાં   59907                   સૌથી  16663
17                    હતું   56599                    તથા  16334
18                      તો   52629                    કામ  16001
19                  દ્વારા   44050                    વાત  15601
20                     નથી   42434                   વર્ષ  15318
21                      જે   40295                  રહેશે  15255
22                      થઈ   39796                આપવામાં  15094
23                       એ   35808                    કરો  14913
24                     હોય   34888                   અન્ય  14616

In [15]:
OUT_PATH = 'gujarati_exp1_preprocessed.csv'

# utf-8-sig so Excel displays Gujarati correctly
df[['id', 'headline', 'text', 'clean_text', 'processed_text']].to_csv(OUT_PATH, index=False, encoding='utf-8-sig')
print('Saved', len(df), 'rows to', OUT_PATH)

Saved 50000 rows to gujarati_exp1_preprocessed.csv


### Step 2: N-Gram Analysis

In [16]:
import math
from indicnlp.tokenize import sentence_tokenize

N_TEST = 1000
MAX_ARTICLES = None   # set e.g. 20000 if RAM runs out while counting trigrams
BOS, EOS, UNK = '<s>', '</s>', '<unk>'


def sentence_tokens(article):
    for sent in sentence_tokenize.sentence_split(article, lang='gu'):
        toks = [normalize_token(t) for t in tokenize(clean_text(sent))]
        toks = [t for t in toks if t]
        if toks:
            yield toks


unigram_counts, bigram_counts, trigram_counts = Counter(), Counter(), Counter()
test_sents = []
n_train_sents = 0

articles = df['text'] if MAX_ARTICLES is None else df['text'].head(MAX_ARTICLES)
for article in tqdm(articles, desc='Counting n-grams'):
    for toks in sentence_tokens(article):
        if len(test_sents) < N_TEST:
            test_sents.append(toks)
            continue
        n_train_sents += 1
        unigram_counts.update(toks)
        unigram_counts[EOS] += 1
        b = [BOS] + toks + [EOS]
        bigram_counts.update(zip(b, b[1:]))
        t = [BOS, BOS] + toks + [EOS]
        trigram_counts.update(zip(t, t[1:], t[2:]))

Counting n-grams: 100%|██████████| 50000/50000 [03:37<00:00, 229.56it/s]


In [17]:
# History (context) counts: C(w1) for bigrams, C(w1, w2) for trigrams
bigram_hist, trigram_hist = Counter(), Counter()
for (w1, _), c in bigram_counts.items():
    bigram_hist[w1] += c
for (w1, w2, _), c in trigram_counts.items():
    trigram_hist[(w1, w2)] += c

V = len(unigram_counts) + 1   # training words + </s> + <unk>

pd.Series({
    'Training sentences': n_train_sents,
    'Test sentences': len(test_sents),
    'Training tokens': sum(unigram_counts.values()) - unigram_counts[EOS],
    'Vocabulary size V (incl. </s>, <unk>)': V,
    'Unique bigrams': len(bigram_counts),
    'Unique trigrams': len(trigram_counts),
}).to_frame('value')

,value
Training sentences,908263
Test sentences,1000
Training tokens,14554292
"Vocabulary size V (incl. </s>, <unk>)",416588
Unique bigrams,4636913
Unique trigrams,9196745


In [18]:
# Top 10 bigrams / trigrams with MLE conditional probability:
#   P(w2 | w1)     = C(w1, w2)     / C(w1)
#   P(w3 | w1, w2) = C(w1, w2, w3) / C(w1, w2)
# N-grams containing the sentence-boundary markers <s>, </s> are skipped.
def top_ngrams(counts, hist, n=10):
    rows = []
    for gram, c in counts.most_common(1000):
        if BOS in gram or EOS in gram:
            continue
        h = gram[0] if len(gram) == 2 else gram[:2]
        rows.append({'ngram': ' '.join(gram), 'frequency': c,
                     'history count': hist[h], 'conditional probability': round(c / hist[h], 4)})
        if len(rows) == n:
            break
    return pd.DataFrame(rows, index=range(1, len(rows) + 1))


top_bi = top_ngrams(bigram_counts, bigram_hist)
top_tri = top_ngrams(trigram_counts, trigram_hist)

print('Top 10 bigrams  –  P(w2 | w1)')
display(top_bi)
print('Top 10 trigrams  –  P(w3 | w1, w2)')
display(top_tri)

Top 10 bigrams  –  P(w2 | w1)


,ngram,frequency,history count,conditional probability
1,છે કે,47393,595096,0.0796
2,છે અને,29890,595096,0.0502
3,રહી છે,24847,33370,0.7446
4,શકે છે,24533,30680,0.7996
5,કરવામાં આવી,19689,59683,0.3299
6,રહ્યા છે,19517,30073,0.6490
7,હતું કે,18366,56457,0.3253
8,આવી છે,18048,64538,0.2796
9,આવે છે,18034,25897,0.6964
10,કરી હતી,16974,100687,0.1686


Top 10 trigrams  –  P(w3 | w1, w2)


,ngram,frequency,history count,conditional probability
1,જણાવ્યું હતું કે,9412,11899,0.7910
2,આ પણ વાંચો,7956,9008,0.8832
3,કરવામાં આવી છે,7570,19689,0.3845
4,કરવામાં આવી હતી,6315,19689,0.3207
5,કહ્યું હતું કે,5050,6142,0.8222
6,કરી રહ્યા છે,5014,6869,0.7299
7,થઈ શકે છે,4262,4977,0.8563
8,આવી રહી છે,4173,4617,0.9038
9,કરી શકે છે,3900,5059,0.7709
10,જણાવી દઈએ કે,3897,3924,0.9931


In [19]:
# Evaluate the 1000 held-out sentences.
# Unsmoothed MLE gives P = 0 to any sentence containing an unseen n-gram, so Laplace (add-one) smoothing is used:
#   P(w2 | w1)     = (C(w1, w2) + 1)     / (C(w1) + V)
#   P(w3 | w1, w2) = (C(w1, w2, w3) + 1) / (C(w1, w2) + V)
# Words not seen in training are mapped to <unk>. Probabilities are summed in log10 space to avoid underflow.
def bigram_prob(w1, w2):
    return (bigram_counts[(w1, w2)] + 1) / (bigram_hist[w1] + V)


def trigram_prob(w1, w2, w3):
    return (trigram_counts[(w1, w2, w3)] + 1) / (trigram_hist[(w1, w2)] + V)


def score_sentence(toks):
    mapped = [w if w in unigram_counts else UNK for w in toks]
    b = [BOS] + mapped + [EOS]
    t = [BOS, BOS] + mapped + [EOS]
    bigrams, trigrams = list(zip(b, b[1:])), list(zip(t, t[1:], t[2:]))
    return {
        'sentence': ' '.join(toks),
        'n_words': len(toks),
        'oov_words': mapped.count(UNK),
        'unseen_bigrams': sum(g not in bigram_counts for g in bigrams),
        'unseen_trigrams': sum(g not in trigram_counts for g in trigrams),
        'log10P_bigram': sum(math.log10(bigram_prob(*g)) for g in bigrams),
        'log10P_trigram': sum(math.log10(trigram_prob(*g)) for g in trigrams),
    }


eval_df = pd.DataFrame([score_sentence(s) for s in test_sents])
n_pred = eval_df['n_words'] + 1   # every word plus </s> is predicted
eval_df['PPL_bigram'] = 10 ** (-eval_df['log10P_bigram'] / n_pred)
eval_df['PPL_trigram'] = 10 ** (-eval_df['log10P_trigram'] / n_pred)

print(f'Test sentences: {len(eval_df)}  |  words: {eval_df["n_words"].sum()}  |  OOV words: {eval_df["oov_words"].sum()}')
for m in ('bigram', 'trigram'):
    corpus_ppl = 10 ** (-eval_df[f'log10P_{m}'].sum() / n_pred.sum())
    zero_mle = (eval_df[f'unseen_{m}s'] > 0).mean() * 100
    print(f'{m:8s}: corpus perplexity = {corpus_ppl:10.1f}  |  '
          f'sentences with P = 0 under unsmoothed MLE: {zero_mle:.1f}%')

eval_df.head(10)

Test sentences: 1000  |  words: 16022  |  OOV words: 352
bigram  : corpus perplexity =    13834.9  |  sentences with P = 0 under unsmoothed MLE: 83.8%
trigram : corpus perplexity =    85171.4  |  sentences with P = 0 under unsmoothed MLE: 90.6%


,sentence,n_words,oov_words,unseen_bigrams,unseen_trigrams,log10P_bigram,log10P_trigram,PPL_bigram,PPL_trigram
0,બાંટવા તા,2,0,1,2,-13.136756,-17.060552,23928.667141,486239.831317
1,બાંટવા ગામે આગામી તા,4,0,1,2,-21.378887,-25.475544,18870.240359,124482.642547
2,રવિવારના રોજ સમય સવારે થી શ્રી લાડી લોહાણા સિંધી પંચાયતની વાડી તળાવ રોડ ગલી નં,15,0,7,11,-76.751196,-85.163262,62654.140202,210234.440231
3,માં લાડી લોહાણા સિંધી પંચાયત બાંટવા દ્વારા આયોજિત લાડી લોહાણા સિંધી નવયુવક મંડળ બાંટવા તેમજ શ્રી લાડી લોહાણા મહિલા મંડળ બાંટવા સહયોગથી જૂનાગઢના નામાંકીત ડોક્ટરો સર્વરોગ નિદાન ફ્રી કેમ્પનું આયોજન ક...,36,0,25,34,-185.843669,-202.560564,105390.594562,298270.189183
4,વિનય હરિયાણી ડો,3,0,1,2,-18.210776,-21.638876,35702.129770,256873.334236
5,મોહિત મારવાણીયા ડો,3,0,2,3,-18.210804,-21.638887,35702.686671,256874.875709
6,હર્ષાબેન ગાધે ડો,3,0,3,4,-19.553098,-22.981287,77316.915051,556316.353006
7,કૃણાલ ભાલોડી ડો,3,0,2,3,-18.138304,-21.566340,34243.330543,246368.400952
8,વિશાલ પટેલ ડો,3,0,0,1,-16.288960,-20.082614,11809.732980,104870.553877
9,પાયલ ભાટુ ડો,3,0,2,3,-17.996999,-21.425021,31568.191463,227119.970241


In [20]:
# Sentence with the highest probability under each model.
# Joint probability always favours short sentences (fewer factors < 1 multiplied together),
# so the best sentence by per-word probability (lowest perplexity, >= 5 words) is shown as well.
def show(title, row, m):
    print(f'{title}')
    print(f'  {row["sentence"]}')
    print(f'  words = {row["n_words"]}  |  log10 P = {row[f"log10P_{m}"]:.3f}  '
          f'(P = {10 ** row[f"log10P_{m}"]:.3e})  |  perplexity = {row[f"PPL_{m}"]:.1f}\n')


long_sents = eval_df[eval_df['n_words'] >= 5]
for m in ('bigram', 'trigram'):
    show(f'[{m}] Highest joint probability:', eval_df.loc[eval_df[f'log10P_{m}'].idxmax()], m)
    show(f'[{m}] Highest per-word probability (>= 5 words):', long_sents.loc[long_sents[f'PPL_{m}'].idxmin()], m)

[bigram] Highest joint probability:
  com
  words = 1  |  log10 P = -5.566  (P = 2.719e-06)  |  perplexity = 606.4

[bigram] Highest per-word probability (>= 5 words):
  પોલીસે તપાસ હાથ ધરી છે
  words = 5  |  log10 P = -12.705  (P = 1.971e-13)  |  perplexity = 131.1

[trigram] Highest joint probability:
  com
  words = 1  |  log10 P = -5.581  (P = 2.626e-06)  |  perplexity = 617.2

[trigram] Highest per-word probability (>= 5 words):
  તમારા ફોન પર લેટેસ્ટ ન્યૂઝ અપડેટ્સ સૌથી પહેલા મેળવવા માટે હમણાં જ sandesh ની મોબાઇલ એપ્લિકેશન ડાઉનલોડ કરો
  words = 18  |  log10 P = -45.737  (P = 1.834e-46)  |  perplexity = 255.4



In [21]:
# Next-word prediction: trigram model on the last two words, backing off to the bigram model on the last word.
# (Scans the n-gram tables, so each call takes a few seconds.)
def predict_next(context, k=5):
    toks = [normalize_token(t) for t in tokenize(clean_text(context))]
    if len(toks) >= 2:
        h = (toks[-2], toks[-1])
        cands = Counter({g[2]: c for g, c in trigram_counts.items() if g[0] == h[0] and g[1] == h[1]})
        if cands:
            return 'trigram', [(w, round(c / trigram_hist[h], 4)) for w, c in cands.most_common(k)]
    if toks:
        h = toks[-1]
        cands = Counter({g[1]: c for g, c in bigram_counts.items() if g[0] == h})
        if cands:
            return 'bigram', [(w, round(c / bigram_hist[h], 4)) for w, c in cands.most_common(k)]
    return None, []


contexts = [
    ' '.join(top_tri.loc[1, 'ngram'].split()[:2]),
    top_bi.loc[1, 'ngram'].split()[0],
    ' '.join(test_sents[1][:3]),
]
for ctx in contexts:
    model, preds = predict_next(ctx)
    print(f'{ctx}  ->  ({model})')
    for w, p in preds:
        print(f'    {w:15s} {p}')

જણાવ્યું હતું  ->  (trigram)
    કે              0.791
    </s>            0.1888
    અને             0.0039
    પરંતુ           0.0014
    જેથી            0.0007
છે  ->  (bigram)
    </s>            0.6807
    કે              0.0796
    અને             0.0502
    જે              0.0147
    ત્યારે          0.0107
બાંટવા ગામે આગામી  ->  (trigram)
    તા              0.5
    મી              0.5


### Step 3: POS Tagging

In [22]:
import numpy as np
from collections import defaultdict


class RuleBasedPOSTagger:
    def __init__(self):
        self.noun_suffixes = ['ો', 'ી', 'ું', 'ા', 'આ', 'એ', 'ઓ', 'ાઓ', 'ીઓ', 'માં', 'ને', 'થી']
        self.verb_suffixes = ['ે', 'યો', 'યા', 'યું', 'શે', 'છે', 'ત', 'વું', 'વી', 'વા', 'તા', 'તી', 'ાવું']
        self.adj_suffixes = ['ીય', 'લી', 'વાળો', 'વાળી', 'વાળું', 'દાર', 'માન']
        self.adv_suffixes = ['થી', 'માં', 'એ']

        self.common_nouns = {
            'ગામ', 'માણસ', 'ઘર', 'શહેર', 'કામ', 'દિવસ', 'રાત', 'સવાર', 'સાંજ',
            'ડોક્ટર', 'ડાક્ટર', 'શિક્ષક', 'વિદ્યાર્થી', 'બાળક', 'સ્ત્રી', 'પુરુષ',
            'પાણી', 'અન્ન', 'કેમ્પ', 'નિદાન', 'સેવા', 'યોજના', 'કાર્યક્રમ', 'પંચાયત',
            'રવિવાર', 'સોમવાર', 'મંગળવાર', 'નામ', 'સ્થળ', 'જગ્યા', 'વિસ્તાર'
        }

        self.common_verbs = {
            'કરવું', 'જવું', 'આવવું', 'થવું', 'રહેવું', 'લખવું', 'વાંચવું', 'બોલવું',
            'યોજાશે', 'કરે', 'કરશે', 'થશે', 'જાય', 'આવે', 'રહે', 'છે', 'હતો', 'હતી',
            'કરી', 'કર્યું', 'થયું', 'આવ્યો', 'ગયો', 'રહ્યો'
        }

        self.common_adj = {
            'સારો', 'નવો', 'જૂનો', 'મોટો', 'નાનો', 'સુંદર', 'સ્વચ્છ', 'ખરાબ',
            'નામાંકિત', 'પ્રખ્યાત', 'પ્રસિદ્ધ', 'લોકપ્રિય', 'મહત્વપૂર્ણ', 'જરૂરી'
        }

        self.common_adv = {
            'અહીં', 'ત્યાં', 'આજે', 'કાલે', 'હવે', 'પછી', 'પહેલાં', 'ફરી', 'હંમેશા',
            'ક્યારેક', 'સદા', 'વારંવાર', 'જલ્દી', 'ધીમે'
        }

        self.common_pronouns = {
            'તે', 'આ', 'હું', 'તું', 'અમે', 'તમે', 'તેઓ', 'આપણે', 'કોઇ', 'શું',
            'કોણ', 'ક્યાં', 'કેવી', 'કેવો', 'તારુ', 'મારુ', 'આપણુ'
        }

        self.common_postpositions = {
            'માં', 'ને', 'થી', 'સુધી', 'માટે', 'સાથે', 'વિના', 'દ્વારા', 'પાસે',
            'ઉપર', 'નીચે', 'આગળ', 'પાછળ', 'વચ્ચે', 'પર'
        }

        self.common_conjunctions = {
            'અને', 'કે', 'પણ', 'કારણ', 'જો', 'તો', 'તેથી', 'માટે', 'અથવા', 'કિંવા'
        }

        self.common_particles = {
            'જ', 'તો', 'ને', 'કે', 'શું', 'ના', 'નહિ', 'નહી'
        }

        self.number_pattern = re.compile(r'^\d+$')

        self.punctuation = {'.', ',', '?', '!', ';', ':', '-', '(', ')', '[', ']', '"', "'"}

    def tag_word(self, word, prev_tag=None):
        """Apply rules to tag a single word"""
        if word in self.punctuation:
            return 'PUNCT'

        if self.number_pattern.match(word):
            return 'NUM'

        if word in self.common_pronouns:
            return 'PRON'
        elif word in self.common_postpositions:
            return 'ADP'
        elif word in self.common_conjunctions:
            return 'CCONJ'
        elif word in self.common_particles:
            return 'PART'
        elif word in self.common_verbs:
            return 'VERB'
        elif word in self.common_adj:
            return 'ADJ'
        elif word in self.common_adv:
            return 'ADV'
        elif word in self.common_nouns:
            return 'NOUN'

        for suffix in self.verb_suffixes:
            if word.endswith(suffix) and len(word) > 2:
                return 'VERB'

        for suffix in self.adj_suffixes:
            if word.endswith(suffix) and len(word) > 3:
                return 'ADJ'

        for suffix in self.adv_suffixes:
            if word.endswith(suffix) and len(word) > 2 and prev_tag in ['VERB', 'ADJ']:
                return 'ADV'

        for suffix in self.noun_suffixes:
            if word.endswith(suffix) and len(word) > 2:
                return 'NOUN'

        if word and word[0].isupper():
            return 'PROPN'

        return 'NOUN'

    def tag_sentence(self, tokens):
        """Tag a token list left to right, passing each predicted tag on as prev_tag"""
        tags, prev_tag = [], None
        for word in tokens:
            prev_tag = self.tag_word(word, prev_tag)
            tags.append(prev_tag)
        return list(zip(tokens, tags))


rule_tagger = RuleBasedPOSTagger()
rule_tagger.tag_sentence('બાંટવા ગામે રવિવારે ડોક્ટરો દ્વારા નિદાન કેમ્પ યોજાશે .'.split())

[('બાંટવા', 'VERB'),
 ('ગામે', 'VERB'),
 ('રવિવારે', 'VERB'),
 ('ડોક્ટરો', 'NOUN'),
 ('દ્વારા', 'ADP'),
 ('નિદાન', 'NOUN'),
 ('કેમ્પ', 'NOUN'),
 ('યોજાશે', 'VERB'),
 ('.', 'PUNCT')]

In [23]:
# POS corpus: sentences tokenized from the raw text WITHOUT the Step 1 cleaning, because the rules
# rely on punctuation (PUNCT), digits (NUM) and capital letters (PROPN), which cleaning removes.
# No gold-annotated Gujarati POS corpus is available, so the rule-based tagger labels the corpus
# ("silver" labels); the HMM is trained on those labels.
N_POS_TRAIN, N_POS_TEST = 20000, 2000


def pos_sentence_tokens(article):
    for sent in sentence_tokenize.sentence_split(article, lang='gu'):
        toks = [normalize_token(t) for t in tokenize(strip_extra_spaces(unicodedata.normalize('NFC', sent)))]
        toks = [t for t in toks if t.strip()]
        if toks:
            yield toks


pos_sents = []
for article in df['text']:
    pos_sents.extend(pos_sentence_tokens(article))
    if len(pos_sents) >= N_POS_TRAIN + N_POS_TEST:
        break

silver = [rule_tagger.tag_sentence(s) for s in pos_sents[:N_POS_TRAIN + N_POS_TEST]]
pos_train, pos_test = silver[:N_POS_TRAIN], silver[N_POS_TRAIN:]
print(f'Train sentences: {len(pos_train)} ({sum(map(len, pos_train))} tokens)  |  '
      f'Test sentences: {len(pos_test)} ({sum(map(len, pos_test))} tokens)')

tag_dist = Counter(t for s in pos_train for _, t in s)
pd.DataFrame(tag_dist.most_common(), columns=['tag', 'count']).assign(
    percent=lambda d: (100 * d['count'] / d['count'].sum()).round(2))

Train sentences: 20000 (370464 tokens)  |  Test sentences: 2000 (37690 tokens)


,tag,count,percent
0,NOUN,194119,52.40
1,VERB,82358,22.23
2,PUNCT,38155,10.30
3,CCONJ,14572,3.93
4,ADP,9919,2.68
5,PRON,7963,2.15
6,NUM,7292,1.97
7,ADV,6612,1.78
8,ADJ,3751,1.01
9,PROPN,3251,0.88


In [24]:
# First-order HMM tagger with Viterbi decoding.
#   Transition  P(t_i | t_{i-1}) : add-one smoothed, with a <START> state
#   Emission    P(w | t)         : add-k smoothed
# Unknown words: training words seen only once are replaced by a word-shape class
# (<NUM>, <PUNCT>, <UPPER>, or <SUF-xx> = last two characters), so the model learns
# P(class | tag) and can tag unseen test words by their shape/suffix.
def word_shape(w):
    if w.isdigit():
        return '<NUM>'
    if all(unicodedata.category(ch)[0] in 'PS' for ch in w):
        return '<PUNCT>'
    if w[0].isupper():
        return '<UPPER>'
    return f'<SUF-{w[-2:]}>'


class HMMTagger:
    def __init__(self, k=0.01):
        self.k = k

    def obs(self, w):
        return w if w in self.known else word_shape(w)

    def fit(self, tagged_sents):
        word_freq = Counter(w for s in tagged_sents for w, _ in s)
        self.known = {w for w, c in word_freq.items() if c > 1}
        self.tags = sorted({t for s in tagged_sents for _, t in s})
        self.tag_idx = {t: i for i, t in enumerate(self.tags)}
        n = len(self.tags)

        trans = np.zeros((n + 1, n))                 # last row = <START>
        emit = defaultdict(lambda: np.zeros(n))
        for s in tagged_sents:
            prev = n
            for w, t in s:
                j = self.tag_idx[t]
                trans[prev, j] += 1
                emit[self.obs(w)][j] += 1
                prev = j

        self.trans_counts = trans
        self.log_trans = np.log((trans + 1) / (trans.sum(axis=1, keepdims=True) + n))
        tag_totals = sum(emit.values())
        denom = tag_totals + self.k * len(emit)
        self.log_emit = {o: np.log((c + self.k) / denom) for o, c in emit.items()}
        self.unseen = np.zeros(n)                    # unseen word class: let transitions decide
        return self

    def tag(self, words):
        emissions = [self.log_emit.get(self.obs(w), self.unseen) for w in words]
        score = self.log_trans[-1] + emissions[0]
        backptrs = []
        for e in emissions[1:]:
            cand = score[:, None] + self.log_trans[:-1]   # cand[i, j]: best path ending in i, then i -> j
            backptrs.append(cand.argmax(axis=0))
            score = cand.max(axis=0) + e
        best = [int(score.argmax())]
        for bp in reversed(backptrs):
            best.append(int(bp[best[-1]]))
        return list(zip(words, [self.tags[i] for i in reversed(best)]))


hmm_tagger = HMMTagger().fit(pos_train)
print('Tags:', hmm_tagger.tags)
print('Known words:', len(hmm_tagger.known), '| emission observations (words + shape classes):', len(hmm_tagger.log_emit))

# Transition probability matrix P(column tag | row tag)
pd.DataFrame(np.exp(hmm_tagger.log_trans), index=hmm_tagger.tags + ['<START>'], columns=hmm_tagger.tags).round(3)

Tags: ['ADJ', 'ADP', 'ADV', 'CCONJ', 'NOUN', 'NUM', 'PART', 'PRON', 'PROPN', 'PUNCT', 'VERB']
Known words: 21070 | emission observations (words + shape classes): 22484


,ADJ,ADP,ADV,CCONJ,NOUN,NUM,PART,PRON,PROPN,PUNCT,VERB
ADJ,0.013,0.012,0.088,0.020,0.547,0.009,0.005,0.007,0.003,0.031,0.265
ADP,0.024,0.007,0.007,0.021,0.662,0.032,0.018,0.030,0.005,0.020,0.175
ADV,0.014,0.008,0.011,0.057,0.517,0.020,0.016,0.035,0.002,0.088,0.232
CCONJ,0.012,0.002,0.015,0.026,0.640,0.018,0.002,0.071,0.006,0.071,0.137
NOUN,0.010,0.036,0.005,0.036,0.572,0.017,0.007,0.009,0.004,0.071,0.234
NUM,0.012,0.048,0.002,0.014,0.662,0.009,0.008,0.001,0.009,0.160,0.077
PART,0.028,0.000,0.003,0.011,0.660,0.009,0.008,0.020,0.015,0.018,0.228
PRON,0.012,0.031,0.021,0.037,0.606,0.008,0.016,0.015,0.003,0.005,0.247
PROPN,0.004,0.026,0.002,0.014,0.338,0.033,0.004,0.003,0.274,0.272,0.031
PUNCT,0.009,0.015,0.006,0.027,0.625,0.071,0.006,0.032,0.041,0.071,0.097


In [25]:
def evaluate(reference_sents, predicted_sents):
    ref = [t for s in reference_sents for _, t in s]
    pred = [t for s in predicted_sents for _, t in s]
    acc = np.mean([r == p for r, p in zip(ref, pred)])
    rows = []
    for tag in sorted(set(ref) | set(pred)):
        tp = sum(r == p == tag for r, p in zip(ref, pred))
        n_pred, n_ref = pred.count(tag), ref.count(tag)
        prec = tp / n_pred if n_pred else 0.0
        rec = tp / n_ref if n_ref else 0.0
        f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
        rows.append({'tag': tag, 'precision': prec, 'recall': rec, 'f1': f1, 'support': n_ref})
    report = pd.DataFrame(rows).set_index('tag').round(3)
    confusion = pd.crosstab(pd.Series(ref, name='reference'), pd.Series(pred, name='predicted'))
    return acc, report, confusion


# HMM on the 2000 held-out sentences, compared with the rule-based labels it was trained to imitate
hmm_test_pred = [hmm_tagger.tag([w for w, _ in s]) for s in tqdm(pos_test, desc='Viterbi')]
acc, report, confusion = evaluate(pos_test, hmm_test_pred)

test_words = [w for s in pos_test for w, _ in s]
oov_rate = np.mean([w not in hmm_tagger.known for w in test_words])
print(f'HMM agreement with rule-based tags on held-out sentences: {acc:.2%}  (unknown-word rate {oov_rate:.2%})')
display(report)
confusion

Viterbi:   0%|          | 0/2000 [00:00<?, ?it/s]

Viterbi: 100%|██████████| 2000/2000 [00:01<00:00, 1135.08it/s]


HMM agreement with rule-based tags on held-out sentences: 99.17%  (unknown-word rate 12.42%)


,precision,recall,f1,support
tag,,,,
ADJ,0.990,0.990,0.990,381
ADP,1.000,1.000,1.000,1044
ADV,0.981,0.700,0.817,751
CCONJ,1.000,1.000,1.000,1621
NOUN,0.987,0.997,0.992,19358
NUM,1.000,1.000,1.000,658
PART,1.000,1.000,1.000,263
PRON,1.000,1.000,1.000,855
PROPN,0.910,0.986,0.946,358


predicted,ADJ,ADP,ADV,CCONJ,NOUN,NUM,PART,PRON,PROPN,PUNCT,VERB
reference,,,,,,,,,,,
ADJ,377,0,0,0,4,0,0,0,0,0,0
ADP,0,1044,0,0,0,0,0,0,0,0,0
ADV,0,0,526,0,224,0,0,0,1,0,0
CCONJ,0,0,0,1621,0,0,0,0,0,0,0
NOUN,4,0,10,0,19306,0,0,0,32,0,6
NUM,0,0,0,0,0,658,0,0,0,0,0
PART,0,0,0,0,0,0,263,0,0,0,0
PRON,0,0,0,0,0,0,0,855,0,0,0
PROPN,0,0,0,0,5,0,0,0,353,0,0


In [26]:
# Agreement with the rule tagger only shows how well the HMM imitates the rules. To measure real tagging
# accuracy, both taggers are scored on a small hand-annotated gold set (news-style sentences, same tagset).
GOLD = '''
રવિવારે/NOUN ગામમાં/NOUN આરોગ્ય/NOUN કેમ્પ/NOUN યોજાશે/VERB ./PUNCT
મુખ્યમંત્રીએ/NOUN આજે/ADV નવી/ADJ યોજનાની/NOUN જાહેરાત/NOUN કરી/VERB ./PUNCT
પોલીસે/NOUN આરોપીની/NOUN ધરપકડ/NOUN કરી/VERB હતી/VERB ./PUNCT
અમદાવાદમાં/PROPN ભારે/ADJ વરસાદ/NOUN પડ્યો/VERB અને/CCONJ રસ્તાઓ/NOUN પર/ADP પાણી/NOUN ભરાયા/VERB ./PUNCT
તેઓ/PRON દરરોજ/ADV આ/PRON મંદિરની/NOUN મુલાકાત/NOUN લે/VERB છે/VERB ./PUNCT
ભારતીય/ADJ ટીમે/NOUN ફાઇનલ/ADJ મેચમાં/NOUN શાનદાર/ADJ જીત/NOUN મેળવી/VERB ./PUNCT
વિદ્યાર્થીઓ/NOUN માટે/ADP શાળામાં/NOUN મફત/ADJ પુસ્તકો/NOUN આપવામાં/VERB આવશે/VERB ./PUNCT
હું/PRON કાલે/ADV મારા/PRON મિત્ર/NOUN સાથે/ADP સુરત/PROPN જઈશ/VERB ./PUNCT
આ/PRON ઘટનામાં/NOUN બે/NUM લોકોના/NOUN મોત/NOUN થયા/VERB છે/VERB ./PUNCT
સરકારે/NOUN ખેડૂતો/NOUN માટે/ADP ૫૦૦/NUM કરોડ/NUM રૂપિયાની/NOUN સહાય/NOUN જાહેર/ADJ કરી/VERB ./PUNCT
શહેરમાં/NOUN કોરોનાના/NOUN નવા/ADJ કેસ/NOUN નોંધાયા/VERB નથી/VERB ./PUNCT
ડોક્ટરે/NOUN દર્દીને/NOUN તરત/ADV જ/PART હોસ્પિટલ/NOUN ખસેડ્યો/VERB ./PUNCT
પ્રધાનમંત્રી/NOUN નરેન્દ્ર/PROPN મોદીએ/PROPN ગાંધીનગરમાં/PROPN નવા/ADJ પુલનું/NOUN ઉદ્ઘાટન/NOUN કર્યું/VERB ./PUNCT
આ/PRON કાર્યક્રમમાં/NOUN મોટી/ADJ સંખ્યામાં/NOUN લોકો/NOUN હાજર/ADJ રહ્યા/VERB હતા/VERB ./PUNCT
તેમણે/PRON કહ્યું/VERB કે/CCONJ આ/PRON નિર્ણય/NOUN ખૂબ/ADV મહત્વપૂર્ણ/ADJ છે/VERB ./PUNCT
IPL/PROPN માં/ADP ગુજરાત/PROPN ટાઇટન્સે/PROPN શાનદાર/ADJ પ્રદર્શન/NOUN કર્યું/VERB ./PUNCT
ખેડૂતો/NOUN ખેતરમાં/NOUN વહેલી/ADJ સવારે/NOUN કામ/NOUN કરે/VERB છે/VERB ,/PUNCT પરંતુ/CCONJ પાણી/NOUN ઓછું/ADJ છે/VERB ./PUNCT
શું/PART તમે/PRON આ/PRON સમાચાર/NOUN વાંચ્યા/VERB ?/PUNCT
શાળાના/NOUN ૧૨/NUM વિદ્યાર્થીઓને/NOUN ઇનામ/NOUN આપવામાં/VERB આવ્યું/VERB ./PUNCT
વડોદરા/PROPN શહેરમાં/NOUN નવી/ADJ બસ/NOUN સેવા/NOUN આવતીકાલથી/ADV મળશે/VERB ./PUNCT
'''

gold_sents = [
    [(normalize_token(unicodedata.normalize('NFC', w)), t) for w, t in (tok.rsplit('/', 1) for tok in line.split())]
    for line in GOLD.strip().splitlines()
]
rule_gold = [rule_tagger.tag_sentence([w for w, _ in s]) for s in gold_sents]
hmm_gold = [hmm_tagger.tag([w for w, _ in s]) for s in gold_sents]

rule_acc, rule_rep, _ = evaluate(gold_sents, rule_gold)
hmm_acc, hmm_rep, _ = evaluate(gold_sents, hmm_gold)
print(f'Gold set: {len(gold_sents)} sentences, {sum(map(len, gold_sents))} tokens')
print(f'Rule-based accuracy: {rule_acc:.2%}')
print(f'HMM accuracy       : {hmm_acc:.2%}')

pd.concat({'Rule-based': rule_rep[['precision', 'recall', 'f1']],
           'HMM': hmm_rep[['precision', 'recall', 'f1']],
           'Gold': rule_rep[['support']]}, axis=1)

Gold set: 20 sentences, 162 tokens
Rule-based accuracy: 70.99%
HMM accuracy       : 70.99%


Rule-based                     HMM                  Gold
       precision recall     f1 precision recall     f1 support
tag                                                           
ADJ        1.000  0.312  0.476     1.000  0.312  0.476      16
ADP        1.000  1.000  1.000     1.000  1.000  1.000       5
ADV        0.500  0.333  0.400     0.500  0.333  0.400       6
CCONJ      1.000  0.667  0.800     1.000  0.667  0.800       3
NOUN       0.667  0.764  0.712     0.667  0.764  0.712      55
NUM        1.000  0.500  0.667     1.000  0.500  0.667       4
PART       1.000  0.500  0.667     1.000  0.500  0.667       2
PRON       0.889  0.800  0.842     0.889  0.800  0.842      10
PROPN      1.000  0.111  0.200     1.000  0.111  0.200       9
PUNCT      1.000  1.000  1.000     1.000  1.000  1.000      21
VERB       0.531  0.839  0.650     0.531  0.839  0.650      31

In [27]:
# Word-by-word comparison on the gold set (X marks a wrong tag)
rows = []
for gs, rs, hs in zip(gold_sents, rule_gold, hmm_gold):
    for (w, g), (_, r), (_, h) in zip(gs, rs, hs):
        rows.append({'word': w, 'gold': g,
                     'rule-based': r + ('' if r == g else ' X'),
                     'HMM': h + ('' if h == g else ' X')})
comparison = pd.DataFrame(rows)

# Show the first few sentences in full, then only the tokens where at least one tagger is wrong
print('Sentence 1-3:')
display(comparison.head(sum(map(len, gold_sents[:3]))))
print('All tokens where at least one tagger disagrees with gold:')
comparison[comparison['rule-based'].str.endswith('X') | comparison['HMM'].str.endswith('X')]

Sentence 1-3:


,word,gold,rule-based,HMM
0,રવિવારે,NOUN,VERB X,VERB X
1,ગામમાં,NOUN,ADV X,ADV X
2,આરોગ્ય,NOUN,NOUN,NOUN
3,કેમ્પ,NOUN,NOUN,NOUN
4,યોજાશે,VERB,VERB,VERB
5,.,PUNCT,PUNCT,PUNCT
6,મુખ્યમંત્રીએ,NOUN,NOUN,NOUN
7,આજે,ADV,ADV,ADV
8,નવી,ADJ,VERB X,VERB X
9,યોજનાની,NOUN,NOUN,NOUN


All tokens where at least one tagger disagrees with gold:


,word,gold,rule-based,HMM
0,રવિવારે,NOUN,VERB X,VERB X
1,ગામમાં,NOUN,ADV X,ADV X
8,નવી,ADJ,VERB X,VERB X
10,જાહેરાત,NOUN,VERB X,VERB X
13,પોલીસે,NOUN,VERB X,VERB X
19,અમદાવાદમાં,PROPN,NOUN X,NOUN X
20,ભારે,ADJ,VERB X,VERB X
30,દરરોજ,ADV,NOUN X,NOUN X
33,મુલાકાત,NOUN,VERB X,VERB X
34,લે,VERB,NOUN X,NOUN X


### Step 4: CFG Parsing

In [28]:
%pip install -q nltk

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [29]:
from itertools import islice
from nltk import CFG, ChartParser, Tree
from nltk.grammar import Nonterminal, Production

# Gujarati is head-final (SOV): arguments, postpositional phrases and adverbs come before the verb group,
# postpositions follow their noun phrase, adjectives/numerals precede the noun, auxiliaries follow the main verb.
# The grammar is written over the Step 3 POS tags (preterminals); the lexical rules (TAG -> 'word')
# are added from whichever tagged sentences are being parsed.
STRUCTURE_RULES = """
S    -> CL | CL PUNCT
CL   -> NP VP | VP | ADVP CL | PART CL | CL CCONJ CL | CL PUNCT CCONJ CL
VP   -> VG | NP VP | PP VP | ADVP VP | ADJP VP | VG COMP
COMP -> CCONJ CL
VG   -> VERB | VERB VG
NP   -> NOM | PRON | PRON NOM | NUMP NOM | PRON NUMP NOM
NOM  -> NOUN | PROPN | NOUN NOM | PROPN NOM | ADJP NOM
ADJP -> ADJ | ADV ADJ
NUMP -> NUM | NUM NUMP
PP   -> NP ADP
ADVP -> ADV | ADV PART
"""
# S = sentence, CL = clause, VP = verb phrase, VG = verb group (main verb + auxiliaries),
# COMP = complement clause introduced by કે, NP = noun phrase, NOM = noun with modifiers / compound noun,
# ADJP = adjective phrase, NUMP = numeral phrase, PP = postpositional phrase, ADVP = adverb phrase
STRUCTURE = CFG.fromstring(STRUCTURE_RULES).productions()
MAX_PARSES = 1000   # cap when enumerating parses of highly ambiguous sentences


def make_parser(tagged_sents):
    lexicon = {Production(Nonterminal(t), [w]) for s in tagged_sents for w, t in s}
    return ChartParser(CFG(Nonterminal('S'), STRUCTURE + sorted(lexicon, key=str)))


def parse(parser, tagged_sent):
    return list(islice(parser.parse([w for w, _ in tagged_sent]), MAX_PARSES))


print(f'{len(STRUCTURE)} structural rules:')
for p in STRUCTURE:
    print('   ', p)

34 structural rules:
    S -> CL
    S -> CL PUNCT
    CL -> NP VP
    CL -> VP
    CL -> ADVP CL
    CL -> PART CL
    CL -> CL CCONJ CL
    CL -> CL PUNCT CCONJ CL
    VP -> VG
    VP -> NP VP
    VP -> PP VP
    VP -> ADVP VP
    VP -> ADJP VP
    VP -> VG COMP
    COMP -> CCONJ CL
    VG -> VERB
    VG -> VERB VG
    NP -> NOM
    NP -> PRON
    NP -> PRON NOM
    NP -> NUMP NOM
    NP -> PRON NUMP NOM
    NOM -> NOUN
    NOM -> PROPN
    NOM -> NOUN NOM
    NOM -> PROPN NOM
    NOM -> ADJP NOM
    ADJP -> ADJ
    ADJP -> ADV ADJ
    NUMP -> NUM
    NUMP -> NUM NUMP
    PP -> NP ADP
    ADVP -> ADV
    ADVP -> ADV PART


In [30]:
# Apply the grammar to the gold-tagged sentences from Step 3
gold_parser = make_parser(gold_sents)
print(f'Grammar: {len(STRUCTURE)} structural + {len(gold_parser.grammar().productions()) - len(STRUCTURE)} lexical rules')

gold_parses = [parse(gold_parser, s) for s in gold_sents]
gold_summary = pd.DataFrame({
    'sentence': [' '.join(w for w, _ in s) for s in gold_sents],
    'tags': [' '.join(t for _, t in s) for s in gold_sents],
    'n_parses': [len(p) for p in gold_parses],
})
print(f"Parsed {(gold_summary['n_parses'] > 0).sum()} / {len(gold_summary)} gold sentences "
      f'(parse counts capped at {MAX_PARSES})')
gold_summary

Grammar: 34 structural + 125 lexical rules
Parsed 20 / 20 gold sentences (parse counts capped at 1000)


,sentence,tags,n_parses
0,રવિવારે ગામમાં આરોગ્ય કેમ્પ યોજાશે .,NOUN NOUN NOUN NOUN VERB PUNCT,16
1,મુખ્યમંત્રીએ આજે નવી યોજનાની જાહેરાત કરી .,NOUN ADV ADJ NOUN NOUN VERB PUNCT,20
2,પોલીસે આરોપીની ધરપકડ કરી હતી .,NOUN NOUN NOUN VERB VERB PUNCT,8
3,અમદાવાદમાં ભારે વરસાદ પડ્યો અને રસ્તાઓ પર પાણી ભરાયા .,PROPN ADJ NOUN VERB CCONJ NOUN ADP NOUN VERB PUNCT,12
4,તેઓ દરરોજ આ મંદિરની મુલાકાત લે છે .,PRON ADV PRON NOUN NOUN VERB VERB PUNCT,8
5,ભારતીય ટીમે ફાઇનલ મેચમાં શાનદાર જીત મેળવી .,ADJ NOUN ADJ NOUN ADJ NOUN VERB PUNCT,27
6,વિદ્યાર્થીઓ માટે શાળામાં મફત પુસ્તકો આપવામાં આવશે .,NOUN ADP NOUN ADJ NOUN VERB VERB PUNCT,3
7,હું કાલે મારા મિત્ર સાથે સુરત જઈશ .,PRON ADV PRON NOUN ADP PROPN VERB PUNCT,4
8,આ ઘટનામાં બે લોકોના મોત થયા છે .,PRON NOUN NUM NOUN NOUN VERB VERB PUNCT,8
9,સરકારે ખેડૂતો માટે ૫૦૦ કરોડ રૂપિયાની સહાય જાહેર કરી .,NOUN NOUN ADP NUM NUM NOUN NOUN ADJ VERB PUNCT,6


In [31]:
# Parse trees: a simple SOV sentence, two coordinated clauses, and a complement (કે) clause
for i in (1, 3, 14):
    print(f"[{i + 1}] {gold_summary.loc[i, 'sentence']}   ({gold_summary.loc[i, 'n_parses']} parses, first shown)")
    if gold_parses[i]:
        gold_parses[i][0].pretty_print()
        print(gold_parses[i][0], '\n')

[2] મુખ્યમંત્રીએ આજે નવી યોજનાની જાહેરાત કરી .   (20 parses, first shown)
                                             S              
                                  ___________|___________    
                                 CL                      |  
                                 |                       |   
                                 VP                      |  
                        _________|_________________      |   
                       NP                          |     |  
                       |                           |     |   
                      NOM                          |     |  
      _________________|___                        |     |   
     |                    NOM                      |     |  
     |             ________|___________            |     |   
     |            |                   NOM          VP    |  
     |            |               _____|_____      |     |   
     |           ADJP            |          NOM    VG    |  
    

In [32]:
# Structural ambiguity: the same tag sequence can be bracketed in several ways,
# e.g. consecutive nouns as one compound NP or as separate arguments of the verb.
i = 0
print(f"{gold_summary.loc[i, 'sentence']}  ->  {gold_summary.loc[i, 'n_parses']} parses. First three:\n")
for t in gold_parses[i][:3]:
    print(t, '\n')

રવિવારે ગામમાં આરોગ્ય કેમ્પ યોજાશે .  ->  16 parses. First three:

(S
  (CL
    (VP
      (NP
        (NOM
          (NOUN રવિવારે)
          (NOM (NOUN ગામમાં) (NOM (NOUN આરોગ્ય) (NOM (NOUN કેમ્પ))))))
      (VP (VG (VERB યોજાશે)))))
  (PUNCT .)) 

(S
  (CL
    (VP
      (NP
        (NOM (NOUN રવિવારે) (NOM (NOUN ગામમાં) (NOM (NOUN આરોગ્ય)))))
      (VP (NP (NOM (NOUN કેમ્પ))) (VP (VG (VERB યોજાશે))))))
  (PUNCT .)) 

(S
  (CL
    (VP
      (NP (NOM (NOUN રવિવારે) (NOM (NOUN ગામમાં))))
      (VP
        (NP (NOM (NOUN આરોગ્ય) (NOM (NOUN કેમ્પ))))
        (VP (VG (VERB યોજાશે))))))
  (PUNCT .)) 



In [33]:
# The grammar accepts Gujarati word order and rejects sequences that violate it
s3, s4 = gold_sents[2], gold_sents[3][5:]
tests = [
    ('SOV order', s3),
    ('verb group moved to the front', s3[3:5] + s3[:3] + s3[5:]),
    ('postposition after its noun', s4),
    ('postposition before its noun', [s4[1], s4[0]] + s4[2:]),
]
for label, s in tests:
    n = len(parse(make_parser([s]), s))
    print(f"{'ACCEPTED' if n else 'REJECTED':8s} | {label:30s} | {' '.join(w for w, _ in s)}")

ACCEPTED | SOV order                      | પોલીસે આરોપીની ધરપકડ કરી હતી .
REJECTED | verb group moved to the front  | કરી હતી પોલીસે આરોપીની ધરપકડ .
ACCEPTED | postposition after its noun    | રસ્તાઓ પર પાણી ભરાયા .
REJECTED | postposition before its noun   | પર રસ્તાઓ પાણી ભરાયા .


In [34]:
# Sentence structure analysis (first parse of each gold sentence): the clause is flattened into its
# ordered constituents, i.e. the arguments/adjuncts followed by the verb group.
def constituents(tree):
    out = []
    for child in tree:
        if isinstance(child, Tree) and child.label() in ('S', 'CL', 'VP'):
            out.extend(constituents(child))
        else:
            out.append(child)
    return out


def count_label(tree, label):
    # maximal subtrees with this label (a VG nested inside a VG counts once)
    return sum(1 for pos in tree.treepositions()
               if isinstance(tree[pos], Tree) and tree[pos].label() == label
               and (len(pos) == 0 or tree[pos[:-1]].label() != label))


rows, rule_use = [], Counter()
for s, parses in zip(gold_sents, gold_parses):
    if not parses:
        continue
    t = parses[0]
    parts = [c for c in constituents(t) if c.label() != 'PUNCT']
    rule_use.update(str(p) for p in t.productions() if p.is_nonlexical())
    rows.append({
        'pattern': ' '.join(c.label() for c in parts),
        'constituents': ' | '.join(f"{c.label()}[{' '.join(c.leaves())}]" for c in parts),
        'verb_final': parts[-1].label() == 'VG',
        'verb_groups': count_label(t, 'VG'),
        'NPs': count_label(t, 'NP'),
        'PPs': count_label(t, 'PP'),
        'tree_height': t.height(),
    })
structure = pd.DataFrame(rows)
print(f"Verb-final sentences: {structure['verb_final'].sum()} / {len(structure)}")
display(structure)
print('Most frequent constituent patterns:')
structure['pattern'].value_counts().head(10)

Verb-final sentences: 20 / 20


,pattern,constituents,verb_final,verb_groups,NPs,PPs,tree_height
0,NP VG,NP[રવિવારે ગામમાં આરોગ્ય કેમ્પ] | VG[યોજાશે],True,1,1,0,10
1,NP VG,NP[મુખ્યમંત્રીએ આજે નવી યોજનાની જાહેરાત] | VG[કરી],True,1,1,0,10
2,NP VG,NP[પોલીસે આરોપીની ધરપકડ] | VG[કરી હતી],True,1,1,0,9
3,NP VG CCONJ PP NP VG,NP[અમદાવાદમાં ભારે વરસાદ] | VG[પડ્યો] | CCONJ[અને] | PP[રસ્તાઓ પર] | NP[પાણી] | VG[ભરાયા],True,2,3,1,10
4,NP ADVP NP VG,NP[તેઓ] | ADVP[દરરોજ] | NP[આ મંદિરની મુલાકાત] | VG[લે છે],True,1,2,0,10
5,ADJP NP VG,ADJP[ભારતીય] | NP[ટીમે ફાઇનલ મેચમાં શાનદાર જીત] | VG[મેળવી],True,1,1,0,12
6,PP NP VG,PP[વિદ્યાર્થીઓ માટે] | NP[શાળામાં મફત પુસ્તકો] | VG[આપવામાં આવશે],True,1,2,1,10
7,NP ADVP PP NP VG,NP[હું] | ADVP[કાલે] | PP[મારા મિત્ર સાથે] | NP[સુરત] | VG[જઈશ],True,1,3,1,10
8,NP NP VG,NP[આ ઘટનામાં] | NP[બે લોકોના મોત] | VG[થયા છે],True,1,2,0,9
9,PP NP ADJP VG,PP[સરકારે ખેડૂતો માટે] | NP[૫૦૦ કરોડ રૂપિયાની સહાય] | ADJP[જાહેર] | VG[કરી],True,1,2,1,9


Most frequent constituent patterns:


pattern
NP VG                          5
NP ADVP NP VG                  2
PP NP VG                       2
NP NP VG                       2
NP VG CCONJ PP NP VG           1
ADJP NP VG                     1
NP ADVP PP NP VG               1
PP NP ADJP VG                  1
NP ADJP VG                     1
NP VG CCONJ NP ADVP ADJP VG    1
Name: count, dtype: int64

In [35]:
# How often each structural rule is used across the gold parses
pd.DataFrame(rule_use.most_common(), columns=['rule', 'times used'])

,rule,times used
0,VP -> NP VP,29
1,NOM -> NOUN,28
2,NOM -> NOUN NOM,27
3,CL -> VP,23
4,VP -> VG,23
5,VG -> VERB,23
6,NP -> NOM,21
7,S -> CL PUNCT,20
8,ADJP -> ADJ,15
9,NOM -> ADJP NOM,11


In [36]:
# Apply the grammar to real corpus sentences (3-10 tokens) tagged by the Step 3 HMM
corpus_tagged = [s for s in hmm_test_pred if 3 <= len(s) <= 10][:300]
res = []
for s in tqdm(corpus_tagged, desc='Parsing'):
    tags = [t for _, t in s]
    content = [t for t in tags if t != 'PUNCT']
    res.append({
        'sentence': ' '.join(w for w, _ in s),
        'tags': ' '.join(tags),
        'n_parses': len(parse(make_parser([s]), s)),
        'ends_in_verb': bool(content) and content[-1] == 'VERB',
        'inner_punct': 'PUNCT' in tags[:-1],
    })
corpus_res = pd.DataFrame(res)
parsed = corpus_res['n_parses'] > 0
print(f'Corpus sentences parsed: {parsed.sum()} / {len(corpus_res)} ({parsed.mean():.1%})')
print(f"Average parses per parsed sentence: {corpus_res.loc[parsed, 'n_parses'].mean():.1f} (capped at {MAX_PARSES})")
print('\nCoverage by sentence property:')
display(pd.crosstab([corpus_res['ends_in_verb'], corpus_res['inner_punct']], parsed.rename('parsed'), margins=True))
print('Examples the grammar could not parse:')
corpus_res.loc[~parsed, ['sentence', 'tags']].head(10)

Parsing: 100%|██████████| 300/300 [00:02<00:00, 121.80it/s]


Corpus sentences parsed: 88 / 300 (29.3%)
Average parses per parsed sentence: 43.8 (capped at 1000)

Coverage by sentence property:


parsed                    False  True  All
ends_in_verb inner_punct                  
False        False           79     0   79
             True            11     0   11
True         False          114    88  202
             True             8     0    8
All                         212    88  300

Examples the grammar could not parse:


,sentence,tags
0,"અમે લગભગ 5,300 ગ્રાહકોને વીજળી વગર જોયા છે .",PRON NOUN NOUN VERB NOUN NOUN VERB VERB PUNCT
1,"પરંતુ ત્યાં સુધી , ફરી , તમે ઘરે રહો .",NOUN ADV ADP PUNCT ADV PUNCT PRON VERB NOUN PUNCT
2,મેટ્રોથી દૂર રહો .,NOUN NOUN NOUN PUNCT
3,રસ્તાઓથી દૂર રહો .,NOUN NOUN NOUN PUNCT
4,આ ભારે પાણીમાં વાહન ન ચલાવો .,PRON VERB NOUN NOUN NOUN NOUN PUNCT
5,કાર્યક્રમનું જીવંત પ્રસારણ કરવામાં આવ્યું હતું .,NOUN VERB NOUN NOUN VERB NOUN PUNCT
6,એક અધિકારીએ કહ્યું અમે મામલાની તપાસ કરી રહ્યા છીએ .,NOUN NOUN VERB PRON NOUN NOUN VERB VERB ADV PUNCT
7,તેની વિગત આપો .,NOUN VERB NOUN PUNCT
9,તેવું શેરબજારનો ટ્રેન્ડ કહી રહ્યો છે .,VERB NOUN NOUN NOUN VERB VERB PUNCT
11,અકસ્માતનું કારણ શું છે તે હજુ સ્પષ્ટ થયું નથી .,NOUN CCONJ PRON VERB PRON NOUN NOUN VERB ADV PUNCT


### Step 5: Probabilistic Parsing

In [37]:
from nltk import PCFG, ViterbiParser
from nltk.grammar import ProbabilisticProduction

MAX_TRAIN_PARSES = 200
N_TRAIN_CANDIDATES = 4000

rule_index = {p: i for i, p in enumerate(STRUCTURE)}
R = len(STRUCTURE)


def rule_counts(tree):
    v = np.zeros(R)
    for p in tree.productions():
        if p.is_nonlexical():
            v[rule_index[p]] += 1
    return v


candidates = [s for s in pos_train if 3 <= len(s) <= 12][:N_TRAIN_CANDIDATES]
forests, no_parse, too_ambiguous = [], 0, 0
for s in tqdm(candidates, desc='Parsing training sentences'):
    trees = list(islice(make_parser([s]).parse([w for w, _ in s]), MAX_TRAIN_PARSES + 1))
    if not trees:
        no_parse += 1
    elif len(trees) > MAX_TRAIN_PARSES:
        too_ambiguous += 1
    else:
        forests.append(np.array([rule_counts(t) for t in trees]))

print(f'Candidates: {len(candidates)} | used for training: {len(forests)} | '
      f'no parse: {no_parse} | > {MAX_TRAIN_PARSES} parses: {too_ambiguous}')
print(f'Total parse trees in training forests: {sum(len(f) for f in forests)}')

Parsing training sentences:   0%|          | 0/4000 [00:00<?, ?it/s]

Parsing training sentences: 100%|██████████| 4000/4000 [00:44<00:00, 90.10it/s] 

Candidates: 4000 | used for training: 748 | no parse: 3201 | > 200 parses: 51
Total parse trees in training forests: 27748


In [38]:
# EM training of the rule probabilities
lhs_of = np.array([str(p.lhs()) for p in STRUCTURE])
lhs_groups = [np.where(lhs_of == a)[0] for a in sorted(set(lhs_of))]
C = np.vstack(forests)                                      # (all parses) x (rules) count matrix
starts = np.cumsum([0] + [len(f) for f in forests[:-1]])    # first row of each sentence's forest
sizes = np.array([len(f) for f in forests])


def normalise(counts, smoothing=0.1):
    theta = np.empty(R)
    for idx in lhs_groups:
        c = counts[idx] + smoothing
        theta[idx] = c / c.sum()
    return theta


def em_step(theta):
    logp = C @ np.log(theta)                                # log P(tree) for every parse
    seg_max = np.maximum.reduceat(logp, starts)
    w = np.exp(logp - np.repeat(seg_max, sizes))
    z = np.add.reduceat(w, starts)
    loglik = np.sum(np.log(z) + seg_max)                    # sum over sentences of log P(sentence)
    posterior = w / np.repeat(z, sizes)                     # P(tree | sentence)
    return normalise(posterior @ C), loglik


uniform = normalise(np.zeros(R), smoothing=1.0)
theta, history = uniform, []
for _ in range(20):
    theta, ll = em_step(theta)
    history.append(ll)

print('Corpus log-likelihood by EM iteration:')
print(pd.Series(history, index=range(1, len(history) + 1)).round(1).to_string())

rule_probs = pd.DataFrame({'rule': [str(p) for p in STRUCTURE], 'uniform': uniform, 'learned': theta}).round(3)
rule_probs

Corpus log-likelihood by EM iteration:
1    -12455.4
2     -5946.0
3     -5850.4
4     -5811.4
5     -5790.6
6     -5777.4
7     -5768.3
8     -5761.6
9     -5756.4
10    -5752.4
11    -5749.1
12    -5746.5
13    -5744.3
14    -5742.4
15    -5740.8
16    -5739.5
17    -5738.3
18    -5737.3
19    -5736.5
20    -5735.7


,rule,uniform,learned
0,S -> CL,0.500,0.003
1,S -> CL PUNCT,0.500,0.997
2,CL -> NP VP,0.167,0.799
3,CL -> VP,0.167,0.173
4,CL -> ADVP CL,0.167,0.022
5,CL -> PART CL,0.167,0.000
6,CL -> CL CCONJ CL,0.167,0.004
7,CL -> CL PUNCT CCONJ CL,0.167,0.001
8,VP -> VG,0.167,0.293
9,VP -> NP VP,0.167,0.593


In [39]:
# PCFG parser for a set of tagged sentences: learned structural probabilities + lexical rules TAG -> word.
# Lexical probabilities are uniform per tag; since each word has one tag in a tagged sentence, they are the
# same for every parse of that sentence and do not affect which tree is most likely.
# ViterbiParser is a CKY-style dynamic-programming parser that returns the single most probable tree.
def make_pcfg_parser(tagged_sents):
    prods = [ProbabilisticProduction(p.lhs(), p.rhs(), prob=theta[i]) for i, p in enumerate(STRUCTURE)]
    words_by_tag = defaultdict(set)
    for s in tagged_sents:
        for w, t in s:
            words_by_tag[t].add(w)
    for t, words in words_by_tag.items():
        prods += [ProbabilisticProduction(Nonterminal(t), [w], prob=1 / len(words)) for w in sorted(words)]
    return ViterbiParser(PCFG(Nonterminal('S'), prods))


def parse_posteriors(parses):
    """P(tree | sentence) for every enumerated parse, highest first."""
    logp = np.array([rule_counts(t) @ np.log(theta) for t in parses])
    post = np.exp(logp - logp.max())
    post /= post.sum()
    order = np.argsort(-post)
    return [parses[i] for i in order], post[order]


def flat(tree):
    return tree.pformat(margin=10 ** 6)

In [40]:
# Most likely parse of each gold sentence
pcfg_gold = make_pcfg_parser(gold_sents)
best_trees, rows = [], []
for s, parses in zip(gold_sents, gold_parses):
    best = next(pcfg_gold.parse([w for w, _ in s]), None)
    best_trees.append(best)
    if best is None:
        rows.append({'sentence': ' '.join(w for w, _ in s), 'n_parses': 0})
        continue
    ranked, post = parse_posteriors(parses)
    rows.append({
        'sentence': ' '.join(w for w, _ in s),
        'n_parses': len(parses),
        'log10 P(best tree)': round(best.logprob() * math.log10(2), 2),   # nltk logprob is base 2
        'P(best | sentence)': round(post[0], 3),
        'best / 2nd best': round(post[0] / post[1], 1) if len(post) > 1 else None,
        'Viterbi = top enumerated': flat(best) == flat(ranked[0]),
        'same as Step 4 first parse': flat(best) == flat(parses[0]),
    })

pcfg_summary = pd.DataFrame(rows)
print(f"Viterbi tree matches the highest-probability enumerated tree: "
      f"{pcfg_summary['Viterbi = top enumerated'].sum()} / {len(pcfg_summary)}")
print(f"PCFG choice differs from the arbitrary first CFG parse of Step 4: "
      f"{(~pcfg_summary['same as Step 4 first parse'].astype(bool)).sum()} / {len(pcfg_summary)}")
pcfg_summary

Viterbi tree matches the highest-probability enumerated tree: 20 / 20
PCFG choice differs from the arbitrary first CFG parse of Step 4: 20 / 20


,sentence,n_parses,log10 P(best tree),P(best | sentence),best / 2nd best,Viterbi = top enumerated,same as Step 4 first parse
0,રવિવારે ગામમાં આરોગ્ય કેમ્પ યોજાશે .,16,-11.19,0.119,1.1,True,False
1,મુખ્યમંત્રીએ આજે નવી યોજનાની જાહેરાત કરી .,20,-14.31,0.188,1.1,True,False
2,પોલીસે આરોપીની ધરપકડ કરી હતી .,8,-10.78,0.233,1.1,True,False
3,અમદાવાદમાં ભારે વરસાદ પડ્યો અને રસ્તાઓ પર પાણી ભરાયા .,12,-21.85,0.478,2.2,True,False
4,તેઓ દરરોજ આ મંદિરની મુલાકાત લે છે .,8,-15.75,0.320,1.1,True,False
5,ભારતીય ટીમે ફાઇનલ મેચમાં શાનદાર જીત મેળવી .,27,-18.18,0.264,2.2,True,False
6,વિદ્યાર્થીઓ માટે શાળામાં મફત પુસ્તકો આપવામાં આવશે .,3,-16.12,0.648,3.6,True,False
7,હું કાલે મારા મિત્ર સાથે સુરત જઈશ .,4,-17.20,0.625,2.4,True,False
8,આ ઘટનામાં બે લોકોના મોત થયા છે .,8,-14.63,0.320,1.1,True,False
9,સરકારે ખેડૂતો માટે ૫૦૦ કરોડ રૂપિયાની સહાય જાહેર કરી .,6,-21.15,0.410,1.1,True,False


In [41]:
# Most likely parse trees for the same three sentences shown in Step 4
for i in (1, 3, 14):
    best = best_trees[i]
    if best is None:
        continue
    print(f"[{i + 1}] {pcfg_summary.loc[i, 'sentence']}")
    print(f"     P(tree) = {best.prob():.3e}   P(tree | sentence) = {pcfg_summary.loc[i, 'P(best | sentence)']}")
    best.pretty_print()
    print(flat(best), '\n')

[2] મુખ્યમંત્રીએ આજે નવી યોજનાની જાહેરાત કરી .
     P(tree) = 4.922e-15   P(tree | sentence) = 0.188
                             S                          
                        _____|_______________________    
                       CL                            |  
      _________________|_____                        |   
     |                       VP                      |  
     |             __________|_______                |   
     |            |                  VP              |  
     |            |           _______|_____          |   
     |            |          |             VP        |  
     |            |          |        _____|___      |   
     NP           |          NP      NP        VP    |  
     |            |          |       |         |     |   
    NOM          ADJP       NOM     NOM        VG    |  
     |         ___|____      |       |         |     |   
    NOUN     ADV      ADJ   NOUN    NOUN      VERB PUNCT
     |        |        |     |       

In [42]:
# Ambiguity resolution: all parses of an ambiguous sentence ranked by probability
def show_ranked(i, k=5):
    ranked, post = parse_posteriors(gold_parses[i])
    print(f"[{i + 1}] {pcfg_summary.loc[i, 'sentence']}  ->  {len(ranked)} parses, top {min(k, len(ranked))}:\n")
    for rank, (t, p) in enumerate(zip(ranked[:k], post[:k]), 1):
        print(f'  #{rank}  P(tree | sentence) = {p:.3f}')
        print(f'      {flat(t)}\n')


ambiguous = pcfg_summary['n_parses'].nlargest(2).index
for i in ambiguous:
    show_ranked(i)

[13] પ્રધાનમંત્રી નરેન્દ્ર મોદીએ ગાંધીનગરમાં નવા પુલનું ઉદ્ઘાટન કર્યું .  ->  96 parses, top 5:

  #1  P(tree | sentence) = 0.052
      (S (CL (NP (NOM (NOUN પ્રધાનમંત્રી))) (VP (NP (NOM (PROPN નરેન્દ્ર) (NOM (PROPN મોદીએ) (NOM (PROPN ગાંધીનગરમાં))))) (VP (ADJP (ADJ નવા)) (VP (NP (NOM (NOUN પુલનું))) (VP (NP (NOM (NOUN ઉદ્ઘાટન))) (VP (VG (VERB કર્યું)))))))) (PUNCT .))

  #2  P(tree | sentence) = 0.050
      (S (CL (NP (NOM (NOUN પ્રધાનમંત્રી) (NOM (PROPN નરેન્દ્ર) (NOM (PROPN મોદીએ) (NOM (PROPN ગાંધીનગરમાં)))))) (VP (ADJP (ADJ નવા)) (VP (NP (NOM (NOUN પુલનું))) (VP (NP (NOM (NOUN ઉદ્ઘાટન))) (VP (VG (VERB કર્યું))))))) (PUNCT .))

  #3  P(tree | sentence) = 0.050
      (S (CL (NP (NOM (NOUN પ્રધાનમંત્રી))) (VP (NP (NOM (PROPN નરેન્દ્ર) (NOM (PROPN મોદીએ) (NOM (PROPN ગાંધીનગરમાં))))) (VP (ADJP (ADJ નવા)) (VP (NP (NOM (NOUN પુલનું) (NOM (NOUN ઉદ્ઘાટન)))) (VP (VG (VERB કર્યું))))))) (PUNCT .))

  #4  P(tree | sentence) = 0.047
      (S (CL (NP (NOM (NOUN પ્રધાનમંત્રી) (NOM (PROPN નરેન્દ્ર

In [43]:
# Apply the PCFG to the HMM-tagged corpus sentences that the CFG parsed in Step 4
corpus_rows = []
for s, n in zip(corpus_tagged, corpus_res['n_parses']):
    if not 0 < n < MAX_PARSES:
        continue
    parses = parse(make_parser([s]), s)
    ranked, post = parse_posteriors(parses)
    corpus_rows.append({
        'sentence': ' '.join(w for w, _ in s),
        'n_parses': len(parses),
        'P(best | sentence)': round(post[0], 3),
        'most likely tree': flat(ranked[0]),
    })
corpus_pcfg = pd.DataFrame(corpus_rows)

print(f'Sentences: {len(corpus_pcfg)}')
print(f"Unambiguous (1 parse): {(corpus_pcfg['n_parses'] == 1).sum()}")
print(f"Mean P(best | sentence) for ambiguous sentences: "
      f"{corpus_pcfg.loc[corpus_pcfg['n_parses'] > 1, 'P(best | sentence)'].mean():.3f}")
print('\nMost ambiguous sentences and the tree the PCFG chooses:')
corpus_pcfg.sort_values('n_parses', ascending=False).head(10)

Sentences: 88
Unambiguous (1 parse): 0
Mean P(best | sentence) for ambiguous sentences: 0.195

Most ambiguous sentences and the tree the PCFG chooses:


,sentence,n_parses,P(best | sentence),most likely tree
82,બોર્ડની વેબસાઈટ પરથી તમામ વિદ્યાર્થીઓ પોતાનું પરિણામ જોઇ શકશે .,256,0.008,(S (CL (NP (NOM (NOUN બોર્ડની))) (VP (NP (NOM (NOUN વેબસાઈટ))) (VP (NP (NOM (NOUN પરથી))) (VP (NP (NOM (NOUN તમામ))) (VP (NP (NOM (NOUN વિદ્યાર્થીઓ))) (VP (NP (NOM (NOUN પોતાનું))) (VP (NP (NOM (N...
37,આ દરમિયાન તેની સરેરાશ માત્ર 20ની આસપાસ રહી છે .,256,0.011,(S (CL (NP (PRON આ) (NOM (NOUN દરમિયાન))) (VP (NP (NOM (NOUN તેની))) (VP (NP (NOM (NOUN સરેરાશ))) (VP (NP (NOM (NOUN માત્ર))) (VP (NP (NOM (NOUN 20ની))) (VP (NP (NOM (NOUN આસપાસ))) (VP (NP (NOM (N...
81,તાલિબાન સામેનો સૌથી મોટો ખતરો ઇસ્લામિક સ્ટેટ જૂથ છે .,192,0.020,(S (CL (NP (NOM (NOUN તાલિબાન))) (VP (NP (NOM (NOUN સામેનો))) (VP (NP (NOM (NOUN સૌથી))) (VP (ADJP (ADJ મોટો)) (VP (NP (NOM (NOUN ખતરો))) (VP (NP (NOM (NOUN ઇસ્લામિક))) (VP (NP (NOM (NOUN સ્ટેટ)))...
31,બધા પહેલા ચરાગવાન વિસ્તારમાં રામુજી વોટર પાર્ક પહોંચ્યા .,128,0.016,(S (CL (NP (NOM (NOUN બધા))) (VP (NP (NOM (NOUN પહેલા))) (VP (NP (NOM (NOUN ચરાગવાન))) (VP (NP (NOM (NOUN વિસ્તારમાં))) (VP (NP (NOM (NOUN રામુજી))) (VP (NP (NOM (NOUN વોટર))) (VP (NP (NOM (NOUN પ...
25,પરંતુ આ દરમિયાન દેશમાં બેરોજગારી ઝડપથી વધી રહી હતી .,128,0.022,(S (CL (NP (NOM (NOUN પરંતુ))) (VP (NP (PRON આ) (NOM (NOUN દરમિયાન))) (VP (NP (NOM (NOUN દેશમાં))) (VP (NP (NOM (NOUN બેરોજગારી))) (VP (NP (NOM (NOUN ઝડપથી))) (VP (NP (NOM (NOUN વધી))) (VP (NP (NO...
34,ઘરવાપસી કરનારા મોટા ભાગના પરિવાર બસના સરાઈપાલીના હતા .,128,0.016,(S (CL (NP (NOM (NOUN ઘરવાપસી))) (VP (NP (NOM (NOUN કરનારા))) (VP (NP (NOM (NOUN મોટા))) (VP (NP (NOM (NOUN ભાગના))) (VP (NP (NOM (NOUN પરિવાર))) (VP (NP (NOM (NOUN બસના))) (VP (NP (NOM (NOUN સરાઈ...
68,વ્યવસાયના દ્રષ્ટિકોણથી ડીઝલનું ઉત્પાદન પેટ્રોલ કરતાં મોંઘું છે .,128,0.016,(S (CL (NP (NOM (NOUN વ્યવસાયના))) (VP (NP (NOM (NOUN દ્રષ્ટિકોણથી))) (VP (NP (NOM (NOUN ડીઝલનું))) (VP (NP (NOM (NOUN ઉત્પાદન))) (VP (NP (NOM (NOUN પેટ્રોલ))) (VP (NP (NOM (NOUN કરતાં))) (VP (NP ...
47,મેયર મત લઈ ગયા તો કેમ કામ નથી કરતા ?,128,0.051,(S (CL (NP (NOM (NOUN મેયર))) (VP (NP (NOM (NOUN મત))) (VP (NP (NOM (NOUN લઈ))) (VP (VG (VERB ગયા)) (COMP (CCONJ તો) (CL (NP (NOM (NOUN કેમ))) (VP (NP (NOM (NOUN કામ))) (VP (NP (NOM (NOUN નથી))) (...
77,આ દર્દનાક અકસ્માતમાં બે અભિનેતાએ પોતાનો જીવ ગુમાવ્યો હતો .,128,0.022,(S (CL (NP (PRON આ) (NOM (NOUN દર્દનાક))) (VP (NP (NOM (NOUN અકસ્માતમાં))) (VP (NP (NOM (NOUN બે))) (VP (NP (NOM (NOUN અભિનેતાએ))) (VP (NP (NOM (NOUN પોતાનો))) (VP (NP (NOM (NOUN જીવ))) (VP (VG (V...
74,સોમનાથના માર્ગો શિવભક્તોનાં જય સોમનાથના નાદથી ગુંજી ઉઠશે .,128,0.016,(S (CL (NP (NOM (NOUN સોમનાથના))) (VP (NP (NOM (NOUN માર્ગો))) (VP (NP (NOM (NOUN શિવભક્તોનાં))) (VP (NP (NOM (NOUN જય))) (VP (NP (NOM (NOUN સોમનાથના))) (VP (NP (NOM (NOUN નાદથી))) (VP (NP (NOM (N...


### Step 6: Semantic Analysis using WordNet

In [44]:
%pip install -q pyiwn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [45]:
# Gujarati WordNet = IndoWordNet, accessed through pyiwn. Importing pyiwn downloads the IndoWordNet
# data (~31 MB) into ~/iwn_data the first time. Semantic relations (hypernymy, hyponymy, ...) are shared
# across the IndoWordNet languages through common synset ids.
import builtins
import pyiwn.iwn as pyiwn_module
from pyiwn import IndoWordNet, Language, PosTag, SynsetRelations


# pyiwn opens its data files without an encoding, so on Windows they are read as cp1252 and the
# Gujarati text fails to decode. Make every open() inside pyiwn default to UTF-8.
def _utf8_open(file, mode='r', *args, **kwargs):
    if 'b' not in mode:
        kwargs.setdefault('encoding', 'utf-8')
    return builtins.open(file, mode, *args, **kwargs)


pyiwn_module.open = _utf8_open

iwn = IndoWordNet(lang=Language.GUJARATI)
WN_WORDS = set(iwn.all_words())

pd.Series({
    'Synsets': len(iwn.all_synsets()),
    'Words (lemmas)': len(WN_WORDS),
    **{f'{p.value.capitalize()} synsets': len(iwn.all_synsets(p)) for p in PosTag},
}).to_frame('Gujarati IndoWordNet')

2026-10-17:13:33:52,860 INFO     [iwn.py:43] Loading gujarati language synsets...


,Gujarati IndoWordNet
Synsets,35599
Words (lemmas),64300
Noun synsets,26503
Verb synsets,2805
Adverb synsets,445
Adjective synsets,5828


In [46]:
# WordNet stores base forms (ઘર, શહેર, મોટો) while news text is inflected (ઘરમાં, શહેરના, મોટા).
# wn_lemma() strips case/plural suffixes and swaps the gender/number endings (-ો/-ું/-ી/-ા)
# until it reaches a form that exists in WordNet.
WN_SUFFIXES = sorted(['માંથી', 'માં', 'નાં', 'ના', 'ની', 'નું', 'નો', 'ને', 'થી', 'એ', 'ઓ', 'ે', 'ો', 'પર'],
                     key=len, reverse=True)
GENDER_ENDINGS = ['ું', 'ો', 'ી', 'ા']


@lru_cache(maxsize=None)
def wn_lemma(word):
    frontier = [word]
    for _ in range(3):
        nxt = []
        for w in frontier:
            if w in WN_WORDS:
                return w
            for e in GENDER_ENDINGS:
                if w.endswith(e):
                    nxt += [w[:-len(e)] + e2 for e2 in GENDER_ENDINGS if e2 != e]
                    break
            nxt += [w[:-len(s)] for s in WN_SUFFIXES if w.endswith(s) and len(w) > len(s) + 1]
        frontier = nxt
    return None


def head_words(synsets, k=None):
    return [s.head_word() for s in synsets][:k]


@lru_cache(maxsize=None)
def hypernyms(synset):
    return tuple(iwn.synset_relation(synset, SynsetRelations.HYPERNYMY))


@lru_cache(maxsize=None)
def hyponyms(synset):
    return tuple(iwn.synset_relation(synset, SynsetRelations.HYPONYMY))


def describe(word, max_senses=3):
    lemma = wn_lemma(word)
    if lemma is None:
        print(f'{word}: not found in IndoWordNet\n')
        return
    senses = iwn.synsets(lemma)
    print(f'{word}  (lemma: {lemma}, {len(senses)} senses)')
    for s in senses[:max_senses]:
        print(f'  [{s.pos()}] {s.gloss()}')
        print(f'      synonyms : {", ".join(n for n in s.lemma_names() if n != lemma) or "-"}')
        if s.examples():
            print(f'      example  : {s.examples()[0]}')
        print(f'      hypernyms: {", ".join(head_words(hypernyms(s))) or "-"}')
        print(f'      hyponyms : {", ".join(head_words(hyponyms(s), 8)) or "-"}')
    print()


for w in ['ઘર', 'પાણી', 'શિક્ષક', 'સારો', 'ખાવું']:
    describe(w)

ઘર  (lemma: ઘર, 12 senses)
  [noun] એ સ્થાન જેમાં કોઇ ભલી-ભાંતિ પરિચિત હોય 
      synonyms : -
      example  : અલ્હાબાદ તો મારે માટે ઘર છે.
      hypernyms: -
      hyponyms : -
  [noun] ઈંટ, પત્થર, લાકડા વગેરેની સ્થાયી રૂપથી બનાવેલી બનાવટ જેમાં છત અને દિવારો હોય છે અને જે વસ્તુની અંતર્ગત આવે છે
      synonyms : ભવન, ઇમારત, મકાન, મહેલ, રહેઠાણ, મુકામ, પ્રાસાદ, વાસ્તુ
      example  : આ ભવનને બનતા ત્રણ વરસ લાગ્યા છે.
      hypernyms: -
      hyponyms : બહુમાળી_ભવન, મિનાર, સચિવાલય, એફિલ_ટાવર, હ્વાઇટ_હાઉસ, વિધાનસભા_ભવન, પ્રસાધન_ગૃહ, બેમાળી
  [noun] માણસો દ્વારા બનાવેલું એ સ્થાન, જે દીવાલોથી ઘેરાયેલું હોય છે
      synonyms : ગૃહ, સદન, આલય, ધામ, નિલય, કેતન, નિકેતન, આયતન, આગર, અવસ્થાન, અવસથ, આશ્રય, રહેઠાણ, સ્થાન
      example  : આ ઘરમાં પાંચ કમરા છે./વિધવા મંગલા નારી નિકેતનમાં રહે છે.
      hypernyms: -
      hyponyms : પ્રાગ્વંશ, ઘોલકું, હાથીખાનું, પ્રસૂતિગૃહ, સ્મશાન, એપાર્ટમેંટ, અતિથિગૃહ, મંદિર

પાણી  (lemma: પાણી, 5 senses)
  [noun] કોઇ વસ્તુ પર રાસાયણિક પ્રક્રિયાથી ચઢાવેલી સોના, ચાદી વગે

In [47]:
# Link the corpus vocabulary (Step 1, stopwords removed) to WordNet
TOP_N = 20000
top_words = final_vocab.most_common(TOP_N)
lemma_freq, direct, via_lemma = Counter(), 0, 0
for w, c in top_words:
    lemma = wn_lemma(w)
    if lemma is None:
        continue
    lemma_freq[lemma] += c
    if lemma == w:
        direct += c
    else:
        via_lemma += c

total = sum(c for _, c in top_words)
n_senses = pd.Series({l: len(iwn.synsets(l)) for l in lemma_freq})
print(f'Token coverage of the top {TOP_N} words: {direct / total:.1%} found directly, '
      f'{via_lemma / total:.1%} after suffix stripping, {1 - (direct + via_lemma) / total:.1%} not in WordNet')
print(f'Distinct WordNet lemmas used in the corpus: {len(lemma_freq)}')
print(f'Average senses per lemma (polysemy): {n_senses.mean():.2f}  |  monosemous lemmas: {(n_senses == 1).mean():.1%}')
print('Most polysemous corpus lemmas:', ', '.join(f'{l} ({n})' for l, n in n_senses.nlargest(10).items()))

Token coverage of the top 20000 words: 45.3% found directly, 29.4% after suffix stripping, 25.3% not in WordNet
Distinct WordNet lemmas used in the corpus: 6392
Average senses per lemma (polysemy): 2.54  |  monosemous lemmas: 41.8%
Most polysemous corpus lemmas: દુ (31), નિ (31), કાઢવું (27), ડોલર (24), પડવું (24), ડૉલર (24), અંત (23), નીકળવું (19), કરવું (18), તેજ (18)


In [48]:
# SYNONYMY: words in the same synset are synonyms. Synonyms per sense of the most frequent corpus lemmas:
rows = []
for lemma, c in lemma_freq.most_common(12):
    senses = iwn.synsets(lemma)
    for s in senses[:2]:
        rows.append({'lemma': lemma, 'corpus freq': c, 'senses': len(senses), 'POS': s.pos(),
                     'synonyms': ', '.join(n for n in s.lemma_names() if n != lemma)[:100],
                     'gloss': s.gloss()[:70]})
pd.DataFrame(rows)

,lemma,corpus freq,senses,POS,synonyms,gloss
0,પોલીસ,43197,2,noun,"પોલીસવાળો, અધિકારી, સૈનિક, સિપાહી, આરક્ષક, આરક્ષી",પ્રજાની જાન-માલની રક્ષા કરનાર સૈનિક કે અધિકારી
1,પોલીસ,43197,2,noun,"પોલીસ_ફોજ, ફોજ","એ વિભાગ, દળ કે સંસ્થા જે પ્રજાના જાન-માલની રક્ષા માટે હોય છે"
2,લોકો,42799,2,noun,"જનતા, લોક, પ્રજાજનો, નાગરિકો",એકથી વધારે વ્યક્તિ
3,લોકો,42799,2,noun,"જનતા, પબ્લિક","લોકોનો એ સમુદાય જે થોડી સામાન્ય રુચિ, મહત્વ ધરાવતો હોય"
4,શક,42092,6,noun,શક_જાતિ,એક પ્રાચીન અનાર્ય જાતિ જે શક દ્વીપની રહેવાસી હતી અને મ્લેચ્છોમાં ગણતી
5,શક,42092,6,noun,"આશંકા, શંકા, સંદેહ, વહેમ, સંશય, ભય, ડર, અંદેશો, અભિશંકા, ખટકો",અશુભની સંભાવનાથી મનમાં થતી કલ્પના
6,વર્ષ,40977,2,noun,"વરસ, સાલ, સંવત્સર",બાર મહિનાનો સમૂહ કે જે સમયની ગણતરી માટેનો એકમ છે
7,વર્ષ,40977,2,noun,"વરસ, સાલ",કોઇ ગ્રહ સૂર્યની પરિક્રમા કરે તે સમયાવધિ
8,દિવસ,35905,8,noun,"દહાડો, દિન, દી, વાર, રોજ, દિવ",સૂર્યોદયથી સૂર્યાસ્ત સુધીનો સમય
9,દિવસ,35905,8,noun,"વાર, દિન, દહાડો",હિન્દુ પંચાંગ પર આધારિત અઠવાડિયાનો કોઈ દિવસ


In [49]:
# Synonyms that actually co-occur in the news corpus: synsets with two or more of their lemmas used
seen, rows = set(), []
for lemma in list(lemma_freq):
    for s in iwn.synsets(lemma):
        if s.synset_id() in seen:
            continue
        seen.add(s.synset_id())
        used = sorted(((n, lemma_freq[n]) for n in s.lemma_names() if lemma_freq[n] > 0), key=lambda x: -x[1])
        if len(used) >= 2:
            rows.append({'synset': s.head_word(), 'POS': s.pos(), 'gloss': s.gloss()[:60],
                         'synonyms used (freq)': ', '.join(f'{n} ({c})' for n, c in used),
                         'total freq': sum(c for _, c in used)})
syn_df = pd.DataFrame(rows).sort_values('total freq', ascending=False).reset_index(drop=True)
print(f'{len(syn_df)} synsets have two or more synonyms used in the corpus')
syn_df.head(15)

3251 synsets have two or more synonyms used in the corpus


,synset,POS,gloss,synonyms used (freq),total freq
0,દિન,noun,એ સમય જેમાં કોઈ વિશેષ વાત હોય,"દિવસ (35905), સમય (31439), દિન (630)",67974
1,પોલીસ,noun,પ્રજાની જાન-માલની રક્ષા કરનાર સૈનિક કે અધિકારી,"પોલીસ (43197), અધિકારી (13106), સૈનિક (1922)",58225
2,દિવસ,noun,સૂર્યોદયથી સૂર્યાસ્ત સુધીનો સમય,"દિવસ (35905), રોજ (11308), વાર (2892), દી (1324), દિન (630)",52059
3,સમય,noun,"મિનિટો, કલાકો, વર્ષો વગેરેમાં મપાતું અંતર અથવા ગતિ જેનાથી ભૂ","સમય (31439), વખત (13612), અમલ (1552), ગાળો (1319), અવસર (973), કાળ (687), યુગ (437), વેળા (294), અવકાશ (257), જમાનો (167), અવધિ (75)",50812
4,લોકો,noun,એકથી વધારે વ્યક્તિ,"લોકો (42799), નાગરિકો (3393), જનતા (2754), લોક (972)",49918
5,ખબર,noun,કોઈ ઉદ્વેશ્યથી કહેલી કે કહેવડાવેલી કે લેખિત કે સાંકેતિક કોઈ,"જાણ (13403), માહિતી (10932), સમાચાર (9176), જાહેરાત (6214), જાણકારી (2970), ખબર (2577), સૂચના (2421), સંદેશ (751), સંદેશો (357)",48801
6,અવસર,noun,કોઈ એવો સમય કે પરિસ્થિતિ કે જેમાં કોઈ કાર્ય કે ઉદ્દેશ સહેલાઈ,"સમય (31439), લાગ (8580), તક (3987), યોગ (2138), અવસર (973), કાળ (687), મોકો (337), વેળા (294), નોબત (93)",48528
7,કુલ,adjective,જેટલું હોય તે બધું,"તમામ (19985), કુલ (10399), સમગ્ર (9196), સંપૂર્ણ (3406), આખું (2537), પૂરું (1817), એકંદર (365), સમસ્ત (271), બધું (255), પરિપૂર્ણ (75), સમ (56)",48362
8,આશંકા,noun,અશુભની સંભાવનાથી મનમાં થતી કલ્પના,"શક (42092), ડર (1528), શંકા (1067), ભય (899), આશંકા (778)",46364
9,જનતા,noun,"લોકોનો એ સમુદાય જે થોડી સામાન્ય રુચિ, મહત્વ ધરાવતો હોય","લોકો (42799), જનતા (2754), પબ્લિક (485)",46038


In [50]:
# HYPERNYMY ("is-a"): follow hypernym links upward from the first noun sense to the top of the hierarchy
def hypernym_chain(synset, max_depth=15):
    chain, seen_ids = [synset], {synset.synset_id()}
    while len(chain) < max_depth:
        parents = [p for p in hypernyms(chain[-1]) if p.synset_id() not in seen_ids]
        if not parents:
            break
        chain.append(parents[0])
        seen_ids.add(parents[0].synset_id())
    return chain


def first_noun_sense(word):
    lemma = wn_lemma(word)
    nouns = iwn.synsets(lemma, pos=PosTag.NOUN) if lemma else []
    return nouns[0] if nouns else None


print('Hypernym chains:')
for w in ['કૂતરો', 'ગાય', 'ગુલાબ', 'શિક્ષક', 'બસ', 'નદી']:
    s = first_noun_sense(w)
    print(f'  {w}: ' + ('  →  '.join(x.head_word() for x in hypernym_chain(s)) if s else 'no noun sense in IndoWordNet'))

print('\nHyponyms (more specific concepts):')
for w in ['પ્રાણી', 'ફળ', 'વાહન']:
    s = first_noun_sense(w)
    if s:
        hypo = hyponyms(s)
        print(f'  {w} ({len(hypo)} hyponyms): {", ".join(head_words(hypo, 15))}')

Hypernym chains:
  કૂતરો: કૂતરો
  ગાય: ગાય
  ગુલાબ: ગુલાબ
  શિક્ષક: અધ્યાપક
  બસ: બસ
  નદી: નદી

Hyponyms (more specific concepts):
  પ્રાણી (46 hyponyms): ઇક્કા, ઈલ, વન્યપશુ, કરોડ_કે_પૃષ્ઠવંશ_વિનાનું, હિંસક, વાસી, એલિયન, માંખીમાર, શિકાર, પૃષ્ઠવંશી, શાકાહારી_પ્રાણી, માંસાહારી_પ્રાણી, સ્થળચર, બચ્ચું, પ્રતિયોગી
  ફળ (26 hyponyms): ઉડનફળ, ટેટો, કોકમ, ખાદ્ય-ફળ, ઇંગોરિયું, લિંબોળી, મગફળી, ડોંડી, અરીઠું, ચીભડું, પથરી, ખિરની, ઝરબેરી, અમ, લસોડું
  વાહન (7 hyponyms): ટેમ્પો, યાન, ગાડી, એક્સપ્રેસ, નરવાહ, રોપ-વે, અપયાન


In [51]:
# Hypernymy-based semantic similarity (path similarity):
#   sim = 1 / (1 + shortest path between two senses through their lowest common hypernym)
@lru_cache(maxsize=None)
def ancestors(synset, max_depth=20):
    dist, by_id, frontier = {synset.synset_id(): 0}, {synset.synset_id(): synset}, [synset]
    for d in range(1, max_depth + 1):
        nxt = []
        for s in frontier:
            for p in hypernyms(s):
                if p.synset_id() not in dist:
                    dist[p.synset_id()] = d
                    by_id[p.synset_id()] = p
                    nxt.append(p)
        frontier = nxt
        if not frontier:
            break
    return dist, by_id


def path_similarity(s1, s2):
    (d1, ids1), (d2, _) = ancestors(s1), ancestors(s2)
    common = set(d1) & set(d2)
    if not common:
        return 0.0, None
    lcs = min(common, key=lambda i: d1[i] + d2[i])
    return 1 / (1 + d1[lcs] + d2[lcs]), ids1[lcs]


def word_similarity(w1, w2, pos=PosTag.NOUN, max_senses=3):
    """Best path similarity over the senses of two words (same POS) and their lowest common hypernym"""
    l1, l2 = wn_lemma(w1), wn_lemma(w2)
    best = (0.0, None)
    if not l1 or not l2:
        return best
    for s1 in iwn.synsets(l1, pos=pos)[:max_senses]:
        for s2 in iwn.synsets(l2, pos=pos)[:max_senses]:
            if s1.pos() == s2.pos():
                best = max(best, path_similarity(s1, s2), key=lambda x: x[0])
    return best


sim_words = ['કૂતરો', 'બિલાડી', 'ગાય', 'ઘોડો', 'શિક્ષક', 'વકીલ', 'નદી', 'પર્વત']
for a, b in [('કૂતરો', 'બિલાડી'), ('કૂતરો', 'શિક્ષક'), ('નદી', 'પર્વત')]:
    score, lcs = word_similarity(a, b)
    print(f'{a} – {b}: similarity {score:.3f}, lowest common hypernym: {lcs.head_word() if lcs else "-"}')

pd.DataFrame([[round(word_similarity(a, b)[0], 3) for b in sim_words] for a in sim_words],
             index=sim_words, columns=sim_words)

કૂતરો – બિલાડી: similarity 0.000, lowest common hypernym: -
કૂતરો – શિક્ષક: similarity 0.000, lowest common hypernym: -
નદી – પર્વત: similarity 0.000, lowest common hypernym: -


,કૂતરો,બિલાડી,ગાય,ઘોડો,શિક્ષક,વકીલ,નદી,પર્વત
કૂતરો,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
બિલાડી,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
ગાય,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
ઘોડો,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
શિક્ષક,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
વકીલ,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
નદી,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
પર્વત,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [52]:
# ANTONYMY. pyiwn does not provide antonymy (Lemma.antonym() is not implemented and the downloaded
# data has no antonym relation), so antonyms are found in two ways:
# 1) Derivational antonyms - Gujarati forms many opposites with negative prefixes:
#    અ-/અન- (સફળ → અસફળ), બિન- (જરૂરી → બિનજરૂરી), ના- (ખુશ → નાખુશ), નિર્- (દોષ → નિર્દોષ),
#    and the સુ-/કુ- (good/bad) pair. A pair is kept only if both words are in WordNet with a shared POS.
NEG_PREFIXES = ['અ', 'અન', 'બિન', 'ના', 'નિર્']


def pos_set(w):
    return {s.pos() for s in iwn.synsets(w)}


def first_gloss(w, pos):
    return next((s.gloss() for s in iwn.synsets(w) if s.pos() == pos), '')


rows = []
for base in WN_WORDS:
    if len(base) < 3:
        continue
    pairs = [(base, pre + base, pre + '-') for pre in NEG_PREFIXES]
    pairs.append(('સુ' + base, 'કુ' + base, 'સુ-/કુ-'))
    for w, ant, rule in pairs:
        if w in WN_WORDS and ant in WN_WORDS:
            shared = pos_set(w) & pos_set(ant)
            if shared:
                pos = sorted(shared)[0]
                rows.append({'word': w, 'antonym': ant, 'rule': rule, 'POS': pos,
                             'freq(word)': lemma_freq[w], 'freq(antonym)': lemma_freq[ant],
                             'gloss(word)': first_gloss(w, pos)[:50], 'gloss(antonym)': first_gloss(ant, pos)[:50]})

deriv_ant = pd.DataFrame(rows)
print(f'Derivational antonym candidates in IndoWordNet: {len(deriv_ant)}')
print(deriv_ant['rule'].value_counts().to_string(), '\n')
print('Pairs where both words occur in the corpus (most frequent first):')
both_used = deriv_ant[(deriv_ant['freq(word)'] > 0) & (deriv_ant['freq(antonym)'] > 0)]
both_used.assign(total=both_used['freq(word)'] + both_used['freq(antonym)']) \
    .sort_values('total', ascending=False).head(15)

Derivational antonym candidates in IndoWordNet: 1141
rule
અ-         1030
ના-          44
નિર્-        27
બિન-         20
સુ-/કુ-      17
અન-           3 

Pairs where both words occur in the corpus (most frequent first):


,word,antonym,rule,POS,freq(word),freq(antonym),gloss(word),gloss(antonym),total
157,ટકા,અટકા,અ-,noun,18897,467,ત્રાંબાનો એક ભારતીય સિક્કો જે આડધા આનાનો હોય છે,માટીનું તે વાસણ જેમાં જગન્નાથપુરીમાં મંદિરના દરવાજ,19364
885,દેશ,નિર્દેશ,નિર્-,noun,13496,942,"પૃથ્વીનો એ ચોક્કસ વિભાગ જેમાં અનેક રાજ્ય, નગર વગેર",(વડીલો દ્વારા નાનાઓને) તે બતાવવાની ક્રિયા કે અમૂક,14438
54,જાણ,અજાણ,અ-,noun,13403,433,કોઈ ઉદ્વેશ્યથી કહેલી કે કહેવડાવેલી કે લેખિત કે સાં,કોઇનાથી ઓળખાણ ન હોવાની અવસ્થા,13836
719,ભાવ,અભાવ,અ-,noun,12039,925,તે નિશ્ચિત કે સ્થિર કરેલું મૂલ્ય જેના પર કોઇ વસ્તુ,ન હોવાની અવસ્થા કે ભાવ,12964
245,નેક,અનેક,અ-,adjective,53,9695,જે બધાની સાથે સારો વ્યવહાર કરતો હોય,એકથી વધારે,9748
943,માતા,નિર્માતા,નિર્-,noun,7438,745,કોઇ પૂજ્ય કે આદરણીય મોટી સ્ત્રી,સ્થાપના કરનારો કે બનાનારો,8183
533,સામાન્ય,અસામાન્ય,અ-,adjective,6489,122,જે ગણનામાં ન હોય,"એવા ગુણ, લક્ષણ વગેરેથી યુક્ત જે કોઇ વર્ગ, પ્રકાર,",6611
408,જરૂરી,બિનજરૂરી,બિન-,adjective,5794,355,જેના વગર સાધારણ રીતે કામ ન ચાલતું હોય,જે કોઈ ઉપયોગી ના હોય કે જે ઉપયોગમાં ના આવે,6149
947,પૂર્વ,અપૂર્વ,અ-,adjective,5822,61,જે પહેલા કોઈ કારણથી તે પદ પર રહી ગયો હોય પણ હવે કો,જેની બરાબરીનું કોઇ ના હોય,5883
779,યોગ્ય,અયોગ્ય,અ-,adjective,4185,222,જેનામાં કોઈ કામ કરવાની શક્તિ કે ગુણ હોય,જે ચૂંટણી કે ચયન માટે અયોગ્ય હોય,4407


In [53]:
# 2) Lexical (non-derivational) antonyms: a curated list of common opposites, checked against WordNet
#    and the corpus. Antonyms are opposite in meaning but usually close in the hierarchy (same hypernym),
#    so the hypernymy-based similarity and lowest common hypernym are shown as well.
LEXICAL_ANTONYMS = [
    ('સારો', 'ખરાબ'), ('મોટો', 'નાનો'), ('નવો', 'જૂનો'), ('ગરમ', 'ઠંડો'), ('ઊંચો', 'નીચો'),
    ('સાચો', 'ખોટો'), ('ગરીબ', 'અમીર'), ('દિવસ', 'રાત'), ('સવાર', 'સાંજ'), ('જીત', 'હાર'),
    ('સુખ', 'દુઃખ'), ('જન્મ', 'મૃત્યુ'), ('શરૂઆત', 'અંત'), ('પ્રેમ', 'નફરત'), ('મિત્ર', 'દુશ્મન'),
    ('લાભ', 'નુકસાન'), ('ખરીદવું', 'વેચવું'), ('આવવું', 'જવું'),
]

rows = []
for a, b in LEXICAL_ANTONYMS:
    la, lb = wn_lemma(a), wn_lemma(b)
    score, lcs = word_similarity(a, b, pos=None)
    rows.append({'word': a, 'antonym': b, 'lemma(word)': la or '-', 'lemma(antonym)': lb or '-',
                 'both in WordNet': bool(la and lb),
                 'freq(word)': lemma_freq[la] if la else 0, 'freq(antonym)': lemma_freq[lb] if lb else 0,
                 'similarity': round(score, 3), 'common hypernym': lcs.head_word() if lcs else '-'})
lex_ant = pd.DataFrame(rows)
print(f"Pairs with both words in IndoWordNet: {lex_ant['both in WordNet'].sum()} / {len(lex_ant)}")
lex_ant

Pairs with both words in IndoWordNet: 18 / 18


,word,antonym,lemma(word),lemma(antonym),both in WordNet,freq(word),freq(antonym),similarity,common hypernym
0,સારો,ખરાબ,સારું,ખરાબ,True,5611,3420,0.0,-
1,મોટો,નાનો,મોટું,નાનો,True,16279,424,0.0,-
2,નવો,જૂનો,નવું,જૂનું,True,25534,2971,0.0,-
3,ગરમ,ઠંડો,ગરમ,ઠંડું,True,1284,550,0.0,-
4,ઊંચો,નીચો,ઊંચું,નીચું,True,1343,916,0.0,-
5,સાચો,ખોટો,સાચો,ખોટો,True,274,379,0.0,-
6,ગરીબ,અમીર,ગરીબ,અમીર,True,1373,277,0.0,-
7,દિવસ,રાત,દિવસ,રાત,True,35905,1870,0.0,-
8,સવાર,સાંજ,સવાર,સાંજ,True,2715,4421,0.0,-
9,જીત,હાર,જીત,હાર,True,4609,1251,0.0,-


In [54]:
# Semantic profile of sentences: lexical relations of every content word (gold tags from Step 3)
ANTONYMS = defaultdict(set)
for df_ant in (deriv_ant, lex_ant):
    for a, b in zip(df_ant['word'], df_ant['antonym']):
        la, lb = wn_lemma(a) or a, wn_lemma(b) or b
        ANTONYMS[la].add(lb)
        ANTONYMS[lb].add(la)

TAG_TO_WN = {'NOUN': PosTag.NOUN, 'VERB': PosTag.VERB, 'ADJ': PosTag.ADJECTIVE, 'ADV': PosTag.ADVERB}


def semantic_profile(tagged_sent):
    rows = []
    for w, t in tagged_sent:
        if t not in TAG_TO_WN:
            continue
        lemma = wn_lemma(w)
        senses = (iwn.synsets(lemma, pos=TAG_TO_WN[t]) or iwn.synsets(lemma)) if lemma else []
        s = senses[0] if senses else None
        rows.append({
            'word': w, 'tag': t, 'lemma': lemma or '-', 'senses': len(senses),
            'synonyms': ', '.join([n for n in s.lemma_names() if n != lemma][:4]) if s else '',
            'hypernym': ', '.join(head_words(hypernyms(s), 2)) if s else '',
            'antonyms': ', '.join(sorted(ANTONYMS.get(lemma, []))) if lemma else '',
            'gloss': s.gloss()[:60] if s else '',
        })
    return pd.DataFrame(rows)


for i in (1, 13):
    print(' '.join(w for w, _ in gold_sents[i]))
    display(semantic_profile(gold_sents[i]))

મુખ્યમંત્રીએ આજે નવી યોજનાની જાહેરાત કરી .


,word,tag,lemma,senses,synonyms,hypernym,antonyms,gloss
0,મુખ્યમંત્રીએ,NOUN,મુખ્યમંત્રી,1,,,,કોઇ રાજ્ય કે પ્રાંતનો મંત્રી જે બીજા બધાં મંત્રીઓમાં પ્રધાન
1,આજે,ADV,આજે,1,"આજ, અદ્ય",,,વર્તમાન દિવસમાં
2,નવી,ADJ,નવું,6,"શિખાઉ, નવશિખાઉ, નવશિખ, કાચું",,જૂનું,જેણે કોઈ કામ હમણાં જ શીખ્યું હોય
3,યોજનાની,NOUN,યોજના,3,"આયોજન, આયોજના, ગોઠવણ",,,કોઈ કામ માટે પેહલેથી કરવામાં આવેલ પ્રબંધ
4,જાહેરાત,NOUN,જાહેરાત,7,"ખબર, જાણ, સંદેશો, સૂચના",,,કોઈ ઉદ્વેશ્યથી કહેલી કે કહેવડાવેલી કે લેખિત કે સાંકેતિક કોઈ
5,કરી,VERB,કરી,3,"હાથી, ગજ, હસ્તી, માંતગ",,,એક શાકાહારી સ્તનપાયી ચોપગું પ્રાણી જે પોતાની સ્થૂળ અને વિશાળ


આ કાર્યક્રમમાં મોટી સંખ્યામાં લોકો હાજર રહ્યા હતા .


,word,tag,lemma,senses,synonyms,hypernym,antonyms,gloss
0,કાર્યક્રમમાં,NOUN,કાર્યક્રમ,2,પ્રોગ્રામ,,,મનોરંજન કે મનોવિનોદને માટે થનારું કોઇ કાર્ય
1,મોટી,ADJ,મોટું,6,"લાંબો, લાંબુ",,નાનો,જે લંબાઈથી યુક્ત હોય
2,સંખ્યામાં,NOUN,સંખ્યા,5,"માપિયું, માપ, માન, માપક",,,તે સાધન જેનાથી કાંઈ માપી શકાય
3,લોકો,NOUN,લોકો,2,"જનતા, લોક, પ્રજાજનો, નાગરિકો",,,એકથી વધારે વ્યક્તિ
4,હાજર,ADJ,હાજર,2,"બાકી, શેષ, અવ્યતીત, અવિગત",,,જે વ્યતીત ન થયું હોય
5,રહ્યા,VERB,-,0,,,,
6,હતા,VERB,હત,1,"ઘવાયેલું, અભિહત",,,જેને માર પડ્યો હોય


### Step 7: Vector Semantics

In [55]:
%pip install -q scikit-learn gensim

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [56]:
from array import array
from scipy import sparse
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize

# Documents = Step 1 output (cleaned, normalized, stopwords removed), already space-separated tokens.
# tokenizer=str.split is essential: sklearn's default token pattern (\b\w\w+\b) would cut Gujarati words
# at every vowel sign, because Python's \w does not match Gujarati matras (same issue as in Step 1).
docs = df['processed_text'].tolist()
headlines = df['headline'].fillna('').tolist()

# BAG OF WORDS: each document is a vector of raw term counts
bow_vectorizer = CountVectorizer(tokenizer=str.split, token_pattern=None, lowercase=False, min_df=5)
X_bow = bow_vectorizer.fit_transform(docs)
bow_vocab = bow_vectorizer.get_feature_names_out()
print(f'Bag-of-words matrix: {X_bow.shape[0]} documents x {X_bow.shape[1]} terms (terms in >= 5 documents)')
print(f'Non-zero entries: {X_bow.nnz} ({X_bow.nnz / (X_bow.shape[0] * X_bow.shape[1]):.4%} of the matrix)')

doc_freq = np.asarray((X_bow > 0).sum(axis=0)).ravel()
print('Highest document-frequency terms:', ', '.join(bow_vocab[i] for i in doc_freq.argsort()[::-1][:20]))

# Bag-of-words vector of the first document (non-zero entries only)
row = X_bow[0]
pd.Series(row.data, index=bow_vocab[row.indices]).sort_values(ascending=False).head(15).to_frame('count')

Bag-of-words matrix: 50000 documents x 87952 terms (terms in >= 5 documents)
Non-zero entries: 6652579 (0.1513% of the matrix)
Highest document-frequency terms: મળી, રીતે, કારણે, ન, બે, શકે, આપી, તમામ, લોકો, કહ્યું, આજે, દરમિયાન, જણાવ્યું, શરૂ, અન્ય, સૌથી, જોવા, વાત, આપવામાં, કામ


,count
ડો,10
બાંટવા,5
લોહાણા,4
શ્રી,3
સિંધી,3
તથા,2
તા,2
મંડળ,2
કાળુભાઇ,1
કૃણાલ,1


In [57]:
# TF-IDF: term frequency weighted by inverse document frequency, so words frequent in one article but
# rare across the corpus get high weight. sublinear_tf uses 1 + log(tf); max_df drops terms that occur
# in more than half of the articles. Rows are L2-normalised.
tfidf_vectorizer = TfidfVectorizer(tokenizer=str.split, token_pattern=None, lowercase=False,
                                   min_df=5, max_df=0.5, sublinear_tf=True)
X_tfidf = tfidf_vectorizer.fit_transform(docs)
tfidf_vocab = tfidf_vectorizer.get_feature_names_out()
idf = pd.Series(tfidf_vectorizer.idf_, index=tfidf_vocab)
print(f'TF-IDF matrix: {X_tfidf.shape[0]} x {X_tfidf.shape[1]}')
print('Lowest IDF (most widespread):', ', '.join(idf.nsmallest(10).index))
print('Highest IDF (rarest kept):   ', ', '.join(idf.nlargest(10).index))


def top_keywords(i, k=8):
    row = X_tfidf[i]
    top = row.indices[np.argsort(row.data)[::-1][:k]]
    return ', '.join(tfidf_vocab[top])


# Keyword extraction: the highest-weighted TF-IDF terms of each article next to its headline
pd.DataFrame({'headline': headlines[:8], 'top TF-IDF keywords': [top_keywords(i) for i in range(8)]})

TF-IDF matrix: 50000 x 87952
Lowest IDF (most widespread): મળી, રીતે, કારણે, ન, બે, શકે, આપી, તમામ, લોકો, કહ્યું
Highest IDF (rarest kept):    acharya, acના, acની, administered, advice, aeronautics, aether, affair, agrawal, ags


,headline,top TF-IDF keywords
0,બાંટવા ગામે રવિવારે જુનાગઢના નામાંકિત ડોકટરો દ્વારા સર્વરોગ નિદાન કેમ્પ યોજાશે,"બાંટવા, લોહાણા, સિંધી, ડો, ભાલોડી, કૈલાશભાઈ, મંડળ, ભાટુ"
1,ગાંધીનગરમાં વરસાદી માહોલ વચ્ચે મચ્છરજન્ય રોગચાળો વકર્યો,"મચ્છરજન્ય, જેનાં, મેલેરિયા, વરસાદી, તાવના, મૌસમ, મચ્છરોનો, ઠેર"
2,WWEની મહિલા સુપરસ્ટાર રેસલરે લીધી નિવૃત્તિ,"wwe, પેજે, khabargujaratlive, પીલી, પેજ, રિંગમાં, ગરદનની, ચાહકો"
3,"હવે આસામમાં બુલડોઝરની એન્ટ્રી, પોલીસ મથકને આગ લગાવનારા પાંચના ઘર તોડી પડાયા","વેપારીને, અટકાયતમાં, બુલડોઝર, તેનુ, ભડકેલા, ઈસ્લામને, મથકમાં, વળતી"
4,16 વર્ષીય બ્રિટિશ પાયલટનું પ્લેન વરસાદમાં ઈંધણ ભરવા માટે અમદાવાદ એરપોર્ટ પર ઉતર્યું,"શાર્ક, પાઇલટ, એરો, પ્લેનમાં, પરિક્રમા, મેક, થાઈલેન્ડ, એકલા"
5,પ્રજાસત્તાક પર્વની પૂર્વ સંધ્યાએ જ કાશ્મીરમાં ગ્રેનેડ એટેક : 4 ઈજાગ્રસ્ત,"રેકગ્નિશન, ફેશિયલ, હાઈટેક, ગ્રેનેડ, સિસ્ટમ, દિલ્હીમાં, વોચ, આતંકવાદીઓ"
6,બાટવા પોલીસ સ્ટેશનના મહિલા PSI અને તેમના સ્ટાફ પર હુમલો,"પાદરડી, પથ્થરમારો, યુવકના, પોલીસની, ગાડીના, કાચ, પીએસઆઈ, પીએસઆઈના"
7,ભારતના ફાળે આજે એકજ દિવસમાં બીજો સિલ્વર : ભારતના નિશાદ કુમારે આજે ટોક્યો પેરાલિમ્પિક્સમાં પુરુષોની હાઇ જમ્પ T47 ઇવેન્ટમાં સિલ્વર મેડલ જીતીને દેશનું ગૌરવ વધાર્યું છે. નિશાદ કુમારે સિલ્વર જીતવા માટે...,"નિશાદ, સિલ્વર, કૂદકો, ઊંચો, કુમારે, ડલ્લાસ, મીટર, પેરાલિમ્પિક્સમાં"


In [58]:
# Document similarity = cosine of document vectors (rows are L2-normalised, so cosine = dot product)
def most_similar_docs(doc_vectors, i, k=5):
    sims = doc_vectors @ doc_vectors[i].T
    sims = sims.toarray().ravel() if sparse.issparse(sims) else np.asarray(sims).ravel()
    sims[i] = -1
    top = np.argsort(sims)[::-1][:k]
    return pd.DataFrame({'doc': top, 'cosine': sims[top].round(3), 'headline': [headlines[j] for j in top]})


QUERY_DOC = 2   # the WWE article from the dataset preview; change to explore other articles
print('Query article:', headlines[QUERY_DOC])
most_similar_docs(X_tfidf, QUERY_DOC)

Query article: WWEની મહિલા સુપરસ્ટાર રેસલરે લીધી નિવૃત્તિ


,doc,cosine,headline
0,6278,0.187,"મેં કોમ્‍પીટીશન દરમિયાન રીંગમાં પ્રવેશ કરતી વખતે ક્‍યારેય અંડરવેર પહેરી ન હતી, મને અંડરવેર પહેરવી પસંત નથીઃ ડબલ્‍યુડબલ્‍યુએફની પૂર્વ મહિલા રેસલર ચેલ્‍સી ગ્રીનનો ખુલાસો"
1,3961,0.143,રાજકોટમાં છેલ્લા 20 દિવસ બાદ પોઝિટિવ કેસ 'શૂન્ય'
2,39591,0.142,માહીનો ફોટો શેર કરી હોલિવૂડનાં દિગ્ગજે એકવાર ફરી સોશિયલમ મીડિયામાં Fans ને કર્યા હેરાન
3,44001,0.140,"કોલકાતામાં બાંગ્લાદેશ હાઇકમીશનની બહાર ગોળીબાર, 2ના મોત"
4,36955,0.138,સુરતમાં વિતેલા 24 કલાકમાં કોરોનાના 10 કેસ નોંધાયા


In [59]:
# LATENT SEMANTIC ANALYSIS: truncated SVD of the TF-IDF matrix gives dense 100-dimensional document
# vectors. Each SVD component is a latent "topic" (a weighted mix of terms).
svd = TruncatedSVD(n_components=100, random_state=42)
X_lsa = normalize(svd.fit_transform(X_tfidf))
print(f'LSA document vectors: {X_lsa.shape[0]} x {X_lsa.shape[1]} | '
      f'variance explained: {svd.explained_variance_ratio_.sum():.1%}')
print('Top terms of the first latent topics (component 0 mostly reflects overall term frequency):')
for c in range(1, 7):
    top = svd.components_[c].argsort()[::-1][:10]
    print(f'  Topic {c}:', ', '.join(tfidf_vocab[top]))

print('\nQuery article:', headlines[QUERY_DOC])
most_similar_docs(X_lsa, QUERY_DOC)

LSA document vectors: 50000 x 100 | variance explained: 8.6%
Top terms of the first latent topics (component 0 mostly reflects overall term frequency):
  Topic 1: પોલીસ, પોલીસે, તપાસ, હાથ, ધરી, ઘટના, ફરિયાદ, જાણ, આરોપી, કાર્યવાહી
  Topic 2: કેસ, કોરોના, નોંધાયા, કોરોનાના, કલાકમાં, કુલ, રાજ્યમાં, નવા, છેલ્લા, ડોઝ
  Topic 3: ગુજરાત, ભાજપ, પ્રમુખ, ચૂંટણી, કોંગ્રેસ, મુખ્યમંત્રી, મંત્રી, સરકાર, જિલ્લા, ખાતે
  Topic 4: રન, મેચ, ટીમ, મેચમાં, વિકેટ, કેપ્ટન, ભારતીય, બનાવ્યા, પ્રથમ, કહ્યું
  Topic 5: વરસાદ, રન, હવામાન, ટીમ, મેચ, વરસાદની, આગાહી, મેચમાં, ભારે, વિકેટ
  Topic 6: રૂપિયા, ભાવ, રૂ, પ્રતિ, ભાવમાં, વધારો, પેટ્રોલ, કિંમત, ટકા, કરોડ

Query article: WWEની મહિલા સુપરસ્ટાર રેસલરે લીધી નિવૃત્તિ


,doc,cosine,headline
0,46159,0.783,નસબંધી-ગર્ભનિરોધક ગોળીઓ બાદ પણ ઘણીવાર પ્રેગ્નેન્ટ થઈ ગઈ આ મહિલા
1,12,0.782,"મેક્સવેલ ટૂંક સમયમાં બનશે ભારતનો જમાઈ, તેની ફિયોન્સેએ આપ્યું અપડેટ"
2,32248,0.781,"એટલું બધુ સેક્સ કર્યું કે બીમાર પડી ગયો વિલ, જાણો શા માટે પ્લેબોય બની ગયો એક્ટર"
3,42140,0.781,આનંદ મહિન્દ્રાએ અગ્નિવીરોને નોકરી આપવાની જાહેરાત કરી
4,27602,0.781,હવે Instagram પર Reels ની જેમ શેર થશે તમામ શોર્ટ વીડિયો


In [60]:
# WORD VECTORS FROM CO-OCCURRENCE: term-term matrix with a +-2 word window, weighted by
# Positive Pointwise Mutual Information:  PPMI(w, c) = max(0, log2( P(w, c) / (P(w) P(c)) ))
# Vocabulary: the 10,000 most frequent words; counted over the first 20,000 articles to keep the loop fast.
PPMI_VOCAB, PPMI_DOCS, WINDOW = 10000, 20000, 2
ppmi_words = [w for w, _ in final_vocab.most_common(PPMI_VOCAB)]
w_idx = {w: i for i, w in enumerate(ppmi_words)}

rows_i, cols_i = array('i'), array('i')
for doc in tqdm(docs[:PPMI_DOCS], desc='Co-occurrence counts'):
    ids = [w_idx.get(t, -1) for t in doc.split()]
    for i, a in enumerate(ids):
        if a < 0:
            continue
        for b in ids[i + 1:i + 1 + WINDOW]:
            if b >= 0:
                rows_i.append(a)
                cols_i.append(b)

r_arr, c_arr = np.frombuffer(rows_i, dtype=np.int32), np.frombuffer(cols_i, dtype=np.int32)
cooc = sparse.coo_matrix((np.ones(len(r_arr)), (r_arr, c_arr)), shape=(PPMI_VOCAB, PPMI_VOCAB)).tocsr()
cooc = cooc + cooc.T                                   # symmetric window counts

n_pairs = cooc.sum()
p_w = np.asarray(cooc.sum(axis=1)).ravel() / n_pairs
p_c = np.asarray(cooc.sum(axis=0)).ravel() / n_pairs
co = cooc.tocoo()
pmi = np.log2((co.data / n_pairs) / (p_w[co.row] * p_c[co.col]))
keep = pmi > 0
ppmi = sparse.csr_matrix((pmi[keep], (co.row[keep], co.col[keep])), shape=cooc.shape)
ppmi_norm = normalize(ppmi)                            # sparse 10,000-dim word vectors
ppmi_svd = normalize(TruncatedSVD(n_components=100, random_state=42).fit_transform(ppmi))   # dense 100-dim

print(f'Co-occurrence matrix: {cooc.shape[0]} x {cooc.shape[1]}, {cooc.nnz} non-zero cells')
print(f'PPMI matrix: {ppmi.nnz} positive cells ({ppmi.nnz / cooc.nnz:.1%} of co-occurring pairs)')


def nearest_words(matrix, word, k=5):
    i = w_idx[word]
    sims = matrix @ matrix[i].T
    sims = sims.toarray().ravel() if sparse.issparse(sims) else np.asarray(sims).ravel()
    sims[i] = -1
    top = np.argsort(sims)[::-1][:k]
    return ', '.join(f'{ppmi_words[j]} ({sims[j]:.2f})' for j in top)


QUERY_WORDS = ['પોલીસ', 'વરસાદ', 'ક્રિકેટ', 'કોરોના', 'ચૂંટણી', 'શાળા', 'ભાજપ', 'હોસ્પિટલ']

Co-occurrence counts: 100%|██████████| 20000/20000 [00:10<00:00, 1881.68it/s]


Co-occurrence matrix: 10000 x 10000, 4000657 non-zero cells
PPMI matrix: 3587393 positive cells (89.7% of co-occurring pairs)


In [61]:
# WORD2VEC (skip-gram with negative sampling): dense word embeddings learned by predicting the context
# words around each word, trained on all 50,000 articles.
from gensim.models import Word2Vec


class DocTokens:
    """Restartable stream of token lists, so the corpus is never held in memory as lists."""
    def __iter__(self):
        for doc in docs:
            yield doc.split()


w2v = Word2Vec(DocTokens(), vector_size=100, window=5, min_count=5, sg=1, negative=5,
               epochs=5, workers=4, seed=42)
wv = w2v.wv
print(f'Word2Vec vocabulary: {len(wv)} words x {wv.vector_size} dimensions')

# Nearest neighbours (cosine) of the same words under the three word representations
query_words = [w for w in QUERY_WORDS if w in w_idx and w in wv]
pd.DataFrame({
    'PPMI (sparse)': [nearest_words(ppmi_norm, w) for w in query_words],
    'PPMI + SVD (100-d)': [nearest_words(ppmi_svd, w) for w in query_words],
    'Word2Vec (100-d)': [', '.join(f'{n} ({s:.2f})' for n, s in wv.most_similar(w, topn=5)) for w in query_words],
}, index=query_words)

2026-10-17:13:36:38,154 INFO     [word2vec.py:582] collecting all words and their counts
2026-10-17:13:36:38,156 INFO     [word2vec.py:565] PROGRESS: at sentence #0, processed 0 words, keeping 0 word types
2026-10-17:13:36:39,765 INFO     [word2vec.py:565] PROGRESS: at sentence #10000, processed 2020410 words, keeping 165467 word types
2026-10-17:13:36:41,916 INFO     [word2vec.py:565] PROGRESS: at sentence #20000, processed 4046392 words, keeping 247633 word types
2026-10-17:13:36:44,53 INFO     [word2vec.py:565] PROGRESS: at sentence #30000, processed 6068250 words, keeping 311941 word types
2026-10-17:13:36:45,846 INFO     [word2vec.py:565] PROGRESS: at sentence #40000, processed 8101397 words, keeping 367984 word types
2026-10-17:13:36:47,729 INFO     [word2vec.py:588] collected 418032 word types from a corpus of 10140610 raw words and 50000 sentences
2026-10-17:13:36:47,731 INFO     [word2vec.py:637] Creating a fresh vocabulary
2026-10-17:13:36:48,415 INFO     [utils.py:447] Word2

Word2Vec vocabulary: 102150 words x 100 dimensions


,PPMI (sparse),PPMI + SVD (100-d),Word2Vec (100-d)
પોલીસ,"પોલીસે (0.46), જાણ (0.37), ફરિયાદ (0.37), પોલીસને (0.36), સ્ટેશનમાં (0.36)","પોલિસ (0.85), ડિવિઝન (0.84), પોલીસના (0.83), પોલીસને (0.81), પોલીસની (0.81)","પોલિસ (0.84), સ્ટેશનના (0.83), પોલીસના (0.81), સ્ટેશનમાં (0.81), પોલીસની (0.77)"
વરસાદ,"ધોધમાર (0.47), ઈંચ (0.45), વરસ્યો (0.44), વરસાદની (0.44), વરસી (0.42)","વરસી (0.95), ધોધમાર (0.94), સાર્વત્રિક (0.92), કમોસમી (0.90), ગાજવીજ (0.90)","વરસ્યો (0.87), ગાજવીજ (0.87), વરસી (0.86), વરસાદની (0.86), ઝાપટા (0.85)"
ક્રિકેટ,"મેચ (0.38), મેચમાં (0.33), ટીમ (0.31), રમાશે (0.30), ipl (0.29)","વર્લ્ડકપ (0.86), રમશે (0.86), ટીમની (0.84), ખેલાડીઓ (0.83), icc (0.83)","આઈસીસી (0.76), icc (0.75), bcci (0.74), ટુર્નામેન્ટ (0.74), બીસીસીઆઈ (0.74)"
કોરોના,"કોવિડ (0.47), કોરોનાના (0.46), વાયરસના (0.36), વાયરસ (0.36), કોરોનાની (0.35)","કોરોનાના (0.93), કોવિડ (0.92), વાયરસના (0.89), કોરોનાની (0.88), કોરોનાનો (0.85)","કોરોનાના (0.88), કોરોનાની (0.87), કોવિડ (0.87), વાયરસના (0.84), વાયરસની (0.83)"
ચૂંટણી,"ચૂંટણીમાં (0.47), વિધાનસભાની (0.45), વિધાનસભા (0.42), બેઠકો (0.40), બેઠક (0.35)","ચૂંટણીમાં (0.94), વિધાનસભા (0.94), વિધાનસભાની (0.92), લોકસભા (0.92), બેઠકો (0.90)","ચૂંટણીઓ (0.90), વિધાનસભાની (0.88), ચુંટણી (0.87), ચૂંટણીના (0.86), ચૂંટણીમાં (0.85)"
શાળા,"શાળાના (0.30), શિક્ષણ (0.26), સ્કૂલ (0.26), શાળામાં (0.25), શાળાઓ (0.25)","શાળામાં (0.92), શાળાના (0.91), શાળાઓમાં (0.86), માધ્યમિક (0.86), પ્રાથમિક (0.84)","શાળાના (0.82), શાળામાં (0.82), શાળાઓમાં (0.82), શાળાને (0.80), શાળાઓ (0.78)"
ભાજપ,"કોંગ્રેસ (0.47), ભાજપના (0.42), પ્રમુખ (0.34), ચૂંટણી (0.34), કોંગ્રેસના (0.33)","કોંગ્રેસ (0.95), ભાજપના (0.89), કોંગ્રેસના (0.88), પાર્ટી (0.86), જનતા (0.86)","કોંગ્રેસ (0.85), ભાજપની (0.85), ભાજપના (0.84), ભાજપા (0.84), ભાજપે (0.80)"
હોસ્પિટલ,"હોસ્પિટલમાં (0.42), સારવાર (0.39), સિવિલ (0.37), અર્થે (0.30), હોસ્પિટલે (0.30)","સિવિલ (0.94), હોસ્પિટલમાં (0.92), સીવીલ (0.88), અર્થે (0.86), હોસ્પિટલે (0.86)","હોસ્પિટલમાં (0.83), હૉસ્પિટલ (0.82), રેફરલ (0.81), હોસ્પીટલ (0.80), હોસ્પીટલમાં (0.79)"


In [62]:
# Word similarity and analogies with Word2Vec.  Analogy a : b :: c : ?  ->  vector(b) - vector(a) + vector(c)
PAIRS = [('પોલીસ', 'ગુનો'), ('વરસાદ', 'પાણી'), ('ક્રિકેટ', 'મેચ'), ('ક્રિકેટ', 'વરસાદ'), ('શાળા', 'વિદ્યાર્થીઓ')]
for a, b in PAIRS:
    if a in wv and b in wv:
        print(f'similarity({a}, {b}) = {wv.similarity(a, b):.3f}')

ANALOGIES = [('ભારત', 'દિલ્હી', 'ગુજરાત'), ('પુરુષ', 'પતિ', 'સ્ત્રી'), ('ભાજપ', 'મોદી', 'કોંગ્રેસ')]
print()
for a, b, c in ANALOGIES:
    if all(w in wv for w in (a, b, c)):
        answers = wv.most_similar(positive=[b, c], negative=[a], topn=3)
        print(f'{a} : {b} :: {c} : ?  ->  ' + ', '.join(f'{w} ({s:.2f})' for w, s in answers))
    else:
        print(f'{a} : {b} :: {c} : ?  ->  skipped (word not in vocabulary)')

similarity(પોલીસ, ગુનો) = 0.606
similarity(વરસાદ, પાણી) = 0.440
similarity(ક્રિકેટ, મેચ) = 0.694
similarity(ક્રિકેટ, વરસાદ) = 0.263
similarity(શાળા, વિદ્યાર્થીઓ) = 0.542

ભારત : દિલ્હી :: ગુજરાત : ?  ->  દિલ્લી (0.63), ચંડીગઢ (0.62), પંજાબ (0.60)
પુરુષ : પતિ :: સ્ત્રી : ?  ->  પત્ની (0.77), સાસુ (0.75), નણંદ (0.73)
ભાજપ : મોદી :: કોંગ્રેસ : ?  ->  નરેન્દ્ર (0.80), પીએમ (0.79), વડાપ્રધાન (0.79)


In [63]:
# Intrinsic evaluation against WordNet (Step 6): synonym pairs should be closer than random word pairs.
# Distributional models also put antonyms close together, because opposites appear in similar contexts.
if 'iwn' not in globals():
    print('Run Step 6 (IndoWordNet) first to evaluate against WordNet synonyms and antonyms.')
else:
    in_both = lambda w: w in w_idx and w in wv
    syn_pairs = set()
    for lemma, _ in lemma_freq.most_common(3000):
        if not in_both(lemma):
            continue
        for s in iwn.synsets(lemma):
            for n in s.lemma_names():
                if n != lemma and in_both(n):
                    syn_pairs.add(tuple(sorted((lemma, n))))
    ant_pairs = {tuple(sorted((a, b))) for d in (deriv_ant, lex_ant)
                 for a, b in zip(d['word'], d['antonym']) if in_both(a) and in_both(b)}
    shared_vocab = [w for w in ppmi_words if w in wv]
    rng = np.random.default_rng(42)
    rand_pairs = {tuple(rng.choice(shared_vocab, 2, replace=False)) for _ in range(1000)}

    cos_ppmi = lambda a, b: float(ppmi_svd[w_idx[a]] @ ppmi_svd[w_idx[b]])
    cos_w2v = lambda a, b: float(wv.similarity(a, b))
    eval_rows = []
    for name, pairs in [('WordNet synonyms', syn_pairs), ('Antonyms (Step 6)', ant_pairs), ('Random pairs', rand_pairs)]:
        eval_rows.append({'pair type': name, 'pairs': len(pairs),
                          'mean cosine PPMI+SVD': np.mean([cos_ppmi(*p) for p in pairs]) if pairs else np.nan,
                          'mean cosine Word2Vec': np.mean([cos_w2v(*p) for p in pairs]) if pairs else np.nan})
    display(pd.DataFrame(eval_rows).round(3))

    examples = sorted(syn_pairs, key=lambda p: -cos_w2v(*p))[:10]
    print('Synonym pairs closest in Word2Vec space:')
    print(', '.join(f'{a}/{b} ({cos_w2v(a, b):.2f})' for a, b in examples))

,pair type,pairs,mean cosine PPMI+SVD,mean cosine Word2Vec
0,WordNet synonyms,4467,0.462,0.393
1,Antonyms (Step 6),38,0.521,0.473
2,Random pairs,1000,0.246,0.220


Synonym pairs closest in Word2Vec space:
માંગણી/માગણી (0.93), રેલવે/રેલ્વે (0.93), શક્યતા/સંભાવના (0.93), કિમી/કિલોમીટર (0.92), અબજ/બિલિયન (0.90), વગર/વિના (0.90), આત્મહત્યા/આપઘાત (0.89), બંને/બન્ને (0.89), નવરાત્રિ/નવરાત્રી (0.89), બેંક/બેન્ક (0.89)


In [64]:
# DOCUMENT VECTORS FROM WORD EMBEDDINGS: average of the Word2Vec vectors of an article's words
k2i = wv.key_to_index


def doc_vector(doc):
    ids = [k2i[t] for t in doc.split() if t in k2i]
    return wv.vectors[ids].mean(axis=0) if ids else np.zeros(wv.vector_size, dtype=np.float32)


X_w2v = normalize(np.vstack([doc_vector(d) for d in tqdm(docs, desc='Averaging word vectors')]))

# The 5 most similar articles to the query article under each document representation
print('Query article:', headlines[QUERY_DOC])
display(pd.DataFrame({name: most_similar_docs(X, QUERY_DOC)['headline'].values
                      for name, X in [('TF-IDF', X_tfidf), ('LSA', X_lsa), ('Average Word2Vec', X_w2v)]},
                     index=range(1, 6)))

pd.DataFrame([
    ('Bag of words', 'document', X_bow.shape[1], 'sparse', 'raw term counts'),
    ('TF-IDF', 'document', X_tfidf.shape[1], 'sparse', 'counts weighted by rarity'),
    ('LSA (SVD of TF-IDF)', 'document', X_lsa.shape[1], 'dense', 'latent topics'),
    ('Average Word2Vec', 'document', X_w2v.shape[1], 'dense', 'mean of word embeddings'),
    ('PPMI', 'word', ppmi.shape[1], 'sparse', 'context-word association'),
    ('PPMI + SVD', 'word', ppmi_svd.shape[1], 'dense', 'compressed PPMI'),
    ('Word2Vec skip-gram', 'word', wv.vector_size, 'dense', 'predictive embedding'),
], columns=['representation', 'unit', 'dimensions', 'type', 'captures'])

Averaging word vectors: 100%|██████████| 50000/50000 [00:05<00:00, 8452.02it/s] 


Query article: WWEની મહિલા સુપરસ્ટાર રેસલરે લીધી નિવૃત્તિ


,TF-IDF,LSA,Average Word2Vec
1,"મેં કોમ્‍પીટીશન દરમિયાન રીંગમાં પ્રવેશ કરતી વખતે ક્‍યારેય અંડરવેર પહેરી ન હતી, મને અંડરવેર પહેરવી પસંત નથીઃ ડબલ્‍યુડબલ્‍યુએફની પૂર્વ મહિલા રેસલર ચેલ્‍સી ગ્રીનનો ખુલાસો",નસબંધી-ગર્ભનિરોધક ગોળીઓ બાદ પણ ઘણીવાર પ્રેગ્નેન્ટ થઈ ગઈ આ મહિલા,"જુબિન નોટિયાલે KBC 13 વિનર હિમાની બુંદેલાને આપી સરપ્રાઇઝ, જુઓ વીડિયો"
2,રાજકોટમાં છેલ્લા 20 દિવસ બાદ પોઝિટિવ કેસ 'શૂન્ય',"મેક્સવેલ ટૂંક સમયમાં બનશે ભારતનો જમાઈ, તેની ફિયોન્સેએ આપ્યું અપડેટ","જીત મેળવ્યા પછી શા માટે ગોલપોસ્ટ પર બેસેલો શ્રીજેશ, આપ્યો આ જવાબ"
3,માહીનો ફોટો શેર કરી હોલિવૂડનાં દિગ્ગજે એકવાર ફરી સોશિયલમ મીડિયામાં Fans ને કર્યા હેરાન,"એટલું બધુ સેક્સ કર્યું કે બીમાર પડી ગયો વિલ, જાણો શા માટે પ્લેબોય બની ગયો એક્ટર","મેક્સવેલ ટૂંક સમયમાં બનશે ભારતનો જમાઈ, તેની ફિયોન્સેએ આપ્યું અપડેટ"
4,"કોલકાતામાં બાંગ્લાદેશ હાઇકમીશનની બહાર ગોળીબાર, 2ના મોત",આનંદ મહિન્દ્રાએ અગ્નિવીરોને નોકરી આપવાની જાહેરાત કરી,કિમ સીઓન હોએ ગેસલાઇટિંગના આરોપો વચ્ચે ભૂતપૂર્વ ગર્લફ્રેન્ડ સાથે સંકળાયેલા વિવાદ માટે માફી માંગી; 2 દિવસ અને 1 રાતનો વિવિધ શો છોડવો: બોલીવુડ સમાચાર
5,સુરતમાં વિતેલા 24 કલાકમાં કોરોનાના 10 કેસ નોંધાયા,હવે Instagram પર Reels ની જેમ શેર થશે તમામ શોર્ટ વીડિયો,"ટાઈમ ટ્રાવેલ કરનારા TikTokerનો દાવો, આવતા વર્ષે એલિયન્સ કરશે ધરતી પર હુમલો!"


,representation,unit,dimensions,type,captures
0,Bag of words,document,87952,sparse,raw term counts
1,TF-IDF,document,87952,sparse,counts weighted by rarity
2,LSA (SVD of TF-IDF),document,100,dense,latent topics
3,Average Word2Vec,document,100,dense,mean of word embeddings
4,PPMI,word,10000,sparse,context-word association
5,PPMI + SVD,word,100,dense,compressed PPMI
6,Word2Vec skip-gram,word,100,dense,predictive embedding


### Step 8: WSD

In [65]:
# WORD EMBEDDINGS FOR WSD. Step 7 trained Word2Vec; here FastText is added. FastText builds a word's
# vector from its character n-grams, so it also gives vectors for words never seen in training. This
# matters for Gujarati, where one lemma appears in many inflected forms (શહેર, શહેરમાં, શહેરના, ...),
# and for WordNet glosses, which contain words that may be rare in news text.
from gensim.models import FastText

ft = FastText(DocTokens(), vector_size=100, window=5, min_count=5, sg=1, min_n=2, max_n=5,
              epochs=5, workers=4, seed=42)
print(f'FastText vocabulary: {len(ft.wv)} words x {ft.wv.vector_size} dimensions')

# Out-of-vocabulary words still get vectors from their character n-grams
for w in ['શહેરોમાંથી', 'ક્રિકેટરોએ', 'વરસાદથી']:
    status = 'in vocabulary' if w in ft.wv.key_to_index else 'OOV, built from character n-grams'
    print(f'{w} ({status}): ' + ', '.join(f'{n} ({s:.2f})' for n, s in ft.wv.most_similar(w, topn=5)))

2026-10-17:13:39:06,534 INFO     [word2vec.py:582] collecting all words and their counts
2026-10-17:13:39:06,535 INFO     [word2vec.py:565] PROGRESS: at sentence #0, processed 0 words, keeping 0 word types
2026-10-17:13:39:07,177 INFO     [word2vec.py:565] PROGRESS: at sentence #10000, processed 2020410 words, keeping 165467 word types
2026-10-17:13:39:07,809 INFO     [word2vec.py:565] PROGRESS: at sentence #20000, processed 4046392 words, keeping 247633 word types
2026-10-17:13:39:08,452 INFO     [word2vec.py:565] PROGRESS: at sentence #30000, processed 6068250 words, keeping 311941 word types
2026-10-17:13:39:09,75 INFO     [word2vec.py:565] PROGRESS: at sentence #40000, processed 8101397 words, keeping 367984 word types
2026-10-17:13:39:09,641 INFO     [word2vec.py:588] collected 418032 word types from a corpus of 10140610 raw words and 50000 sentences
2026-10-17:13:39:09,641 INFO     [word2vec.py:637] Creating a fresh vocabulary
2026-10-17:13:39:09,925 INFO     [utils.py:447] FastT

FastText vocabulary: 102150 words x 100 dimensions
શહેરોમાંથી (in vocabulary): શહેરોમાં (0.88), શહેરો (0.86), શહેરોથી (0.83), શહેરોએ (0.83), શહેરોમાંનું (0.82)
ક્રિકેટરોએ (in vocabulary): ક્રિકેટરે (0.94), ક્રિકેટરો (0.93), ક્રિકેટરોને (0.92), ક્રિકેટરોના (0.92), ક્રિકેટરોમાં (0.91)
વરસાદથી (in vocabulary): વરસાદૃી (0.91), વરસાદૃ (0.91), વરસાદના (0.90), વરસાદમા (0.90), વરસાદનાં (0.89)


In [66]:
# Static embeddings give ONE vector per word, so all senses of a polysemous word are mixed together.
# Frequent polysemous corpus words (>= 3 IndoWordNet senses), their senses and their FastText neighbours:
ambiguous = [l for l, _ in lemma_freq.most_common(2000)
             if len(l) >= 3 and l in ft.wv.key_to_index and len(iwn.synsets(l)) >= 3][:8]
for lemma in ambiguous:
    senses = iwn.synsets(lemma)
    print(f'{lemma}  ({len(senses)} senses, corpus frequency {lemma_freq[lemma]})')
    for s in senses[:4]:
        print(f'    [{s.pos()}] {s.gloss()[:80]}')
    print('    FastText neighbours:', ', '.join(n for n, _ in ft.wv.most_similar(lemma, topn=8)), '\n')

દિવસ  (8 senses, corpus frequency 35905)
    [noun] સૂર્યોદયથી સૂર્યાસ્ત સુધીનો સમય
    [noun] હિન્દુ પંચાંગ પર આધારિત અઠવાડિયાનો કોઈ દિવસ
    [noun] એક સૂર્યોદયથી બીજા સૂર્યોદય સુધીનો સમય જે ચોવીસ કલાકનો મનાય છે
    [noun] નિશ્ચિત કે ઉચિત સમય
    FastText neighbours: એકદિવસ, દિવસો, દિવસને, રાતદિવસ, દિવસનુ, દિવસનો, દિવસનાં, દિવસની 

સમય  (9 senses, corpus frequency 31439)
    [noun] મિનિટો, કલાકો, વર્ષો વગેરેમાં મપાતું અંતર અથવા ગતિ જેનાથી ભૂત, વર્તમાન વગેરેનો બ
    [noun] એ સમય કે જે કોઇને વિશેષ અવસ્થામાં કોઇ કાર્ય કરવા કે પોતાની ફરજ પૂરી કરવા માટે મળ
    [noun] કોઈ એવો સમય કે પરિસ્થિતિ કે જેમાં કોઈ કાર્ય કે ઉદ્દેશ સહેલાઈથી અને જલ્દી કે સહેલ
    [noun] એ સમય જેમાં કોઈ વિશેષ વાત હોય
    FastText neighbours: થોડોક, વિતાવો, થોડો, સમયઃ, સમયને, સમયગાળો, થોડાંક, લાંબો 

આપવું  (7 senses, corpus frequency 27672)
    [verb] ભાડે કે ભાડા પર આપવું
    [verb] ઉપલબ્ધ કે સુલભ કરાવું
    [verb] મૌખિક રૂપે પ્રસ્તુત કરવું
    [verb] ચૂકવણી કરવા કે આપવાનો પ્રસ્તાવ રાખવો કે કામના બદલામાં ધન પ્રસ્તુત કર

In [67]:
# WORD SENSE DISAMBIGUATION: pick the IndoWordNet sense of a word that best fits its context.
# Sense signature = words of the sense's gloss + example sentences + synonyms.
#   first    : first sense listed in IndoWordNet (baseline; IndoWordNet has no sense frequencies)
#   lesk     : simplified Lesk, number of (lemmatised) words shared by the context and the signature
#   word2vec : cosine between the averaged Word2Vec vectors of the context and of the signature
#   fasttext : the same with FastText vectors (also covers OOV words)
#   lesk+emb : Lesk overlap, with ties broken by the FastText cosine
# In every method a tie falls back to the earlier-listed sense.
METHODS = ['first', 'lesk', 'word2vec', 'fasttext', 'lesk+emb']


@lru_cache(maxsize=None)
def prep(text):
    return tuple(remove_stopwords([normalize_token(t) for t in tokenize(clean_text(text))]))


def lem(t):
    return wn_lemma(t) or t


def signature(synset, lemma, exclude_example=None):
    words = list(prep(synset.gloss()))
    for n in synset.lemma_names():
        words += prep(n.replace('_', ' '))
    for ex in synset.examples():
        if ex != exclude_example:
            words += prep(ex)
    return [w for w in words if lem(w) != lemma]   # the target word itself does not discriminate


@lru_cache(maxsize=None)
def token_vec(model, t):
    if model == 'fasttext':
        return ft.wv[t]
    return wv[t] if t in wv.key_to_index else None


def mean_vec(model, tokens):
    vecs = [v for v in (token_vec(model, t) for t in tokens) if v is not None]
    if not vecs:
        return None
    v = np.mean(vecs, axis=0)
    n = np.linalg.norm(v)
    return v / n if n > 0 else None


def cosines(model, ctx, sigs):
    c = mean_vec(model, ctx)
    out = []
    for sig in sigs:
        s = mean_vec(model, sig)
        out.append(float(c @ s) if c is not None and s is not None else -1.0)
    return np.array(out)


def score_senses(lemma, context_tokens, senses, method, exclude_example=None):
    if method == 'first':
        return np.array([1.0] + [0.0] * (len(senses) - 1))
    ctx = [t for t in context_tokens if lem(t) != lemma]
    sigs = [signature(s, lemma, exclude_example) for s in senses]
    if method in ('word2vec', 'fasttext'):
        return cosines(method, ctx, sigs)
    ctx_lemmas = {lem(t) for t in ctx}
    overlap = np.array([len(ctx_lemmas & {lem(t) for t in sig}) for sig in sigs], dtype=float)
    if method == 'lesk':
        return overlap
    return overlap + 0.1 * cosines('fasttext', ctx, sigs)   # |0.1 * cosine| < 1, so it only breaks ties


def disambiguate(lemma, context_tokens, method='lesk+emb', exclude_example=None):
    senses = iwn.synsets(lemma)
    if not senses:
        return None, []
    scores = score_senses(lemma, context_tokens, senses, method, exclude_example)
    return senses[int(np.argmax(scores))], list(zip(senses, scores))

In [68]:
# Evaluation with sense-annotated data taken from IndoWordNet itself: each example sentence of a synset
# shows the word used in that sense. For polysemous corpus lemmas every example becomes a test case
# (context = the example sentence, gold = its synset). The example is removed from the sense signatures
# (leave-one-out), so the answer is never visible to the disambiguator.
MAX_CASES = 3000
test_cases = []
for lemma, _ in lemma_freq.most_common(5000):
    senses = iwn.synsets(lemma)
    if len(lemma) < 3 or len(senses) < 2:
        continue
    for s in senses:
        for ex in s.examples():
            if prep(ex):
                test_cases.append((lemma, ex, s.synset_id(), len(senses)))
test_cases = test_cases[:MAX_CASES]
print(f'Test cases: {len(test_cases)} example sentences of {len({c[0] for c in test_cases})} polysemous lemmas')

results = {m: [] for m in METHODS}
for lemma, ex, gold_id, _ in tqdm(test_cases, desc='Disambiguating'):
    for m in METHODS:
        pred, _ = disambiguate(lemma, prep(ex), m, exclude_example=ex)
        results[m].append(pred.synset_id() == gold_id)

n_senses_case = np.array([c[3] for c in test_cases])
bucket = pd.cut(n_senses_case, [1, 2, 4, 10 ** 6], labels=['2 senses', '3-4 senses', '5+ senses'])
acc = pd.DataFrame({m: results[m] for m in METHODS}, dtype=float)
acc['random'] = 1 / n_senses_case          # expected accuracy of a random guess
wsd_summary = pd.concat([acc.mean().rename('all cases'), acc.groupby(bucket, observed=True).mean().T], axis=1)

best_method = wsd_summary['all cases'].drop(['random', 'first']).idxmax()
print('Cases per group:', pd.Series(bucket).value_counts().sort_index().to_dict())
print(f'Best method: {best_method}')
wsd_summary.round(3)

Test cases: 3000 example sentences of 677 polysemous lemmas


Disambiguating: 100%|██████████| 3000/3000 [00:23<00:00, 126.41it/s]

Cases per group: {'2 senses': 411, '3-4 senses': 822, '5+ senses': 1767}
Best method: lesk+emb


,all cases,2 senses,3-4 senses,5+ senses
first,0.226,0.501,0.294,0.131
lesk,0.461,0.637,0.538,0.384
word2vec,0.336,0.450,0.425,0.269
fasttext,0.369,0.518,0.437,0.302
lesk+emb,0.468,0.620,0.528,0.405
random,0.225,0.500,0.293,0.130


In [69]:
# The same word in different contexts: scores of every sense (→ marks the chosen sense)
EXAMPLES = [
    ('હાર', 'ભારતીય ટીમની ફાઇનલ મેચમાં હાર થઈ અને ચાહકો નિરાશ થયા'),
    ('હાર', 'લગ્નમાં કન્યાએ સોનાનો સુંદર હાર પહેર્યો હતો'),
    ('કલમ', 'પોલીસે આરોપી સામે કાયદાની કલમ હેઠળ ગુનો નોંધ્યો'),
    ('કલમ', 'વિદ્યાર્થીએ નવી કલમથી પરીક્ષાના જવાબ લખ્યા'),
]
for word, sent in EXAMPLES:
    lemma = wn_lemma(word)
    if not lemma:
        print(f'{word}: not in IndoWordNet\n')
        continue
    best, scored = disambiguate(lemma, prep(sent), best_method)
    print(f'{sent}   [target: {lemma}, method: {best_method}]')
    for s, sc in scored:
        print(f"  {'→' if s is best else ' '} {sc:6.3f}  [{s.pos()}] {s.gloss()[:75]}")
    print()

ભારતીય ટીમની ફાઇનલ મેચમાં હાર થઈ અને ચાહકો નિરાશ થયા   [target: હાર, method: lesk+emb]
     0.069  [noun] એક શેરની માળા
     0.043  [noun] ફૂલોની ગૂંથેલી માળા
  →  0.073  [noun] પરાજિત થવાની અવસ્થા કે ભાવ
     0.045  [noun] સુતર વગેરેમાં ગોળાકાર પરોવેલી કોઇ વસ્તુ જેમકે મણકા, ફૂલ આદિ જે ગળામાં પહેરવ
     0.064  [noun] એવી પરંપરા જેમાં એક જ પ્રકારની વસ્તુઓ, વ્યક્તિઓ કે જીવો એકબીજાની પાછળ એક કત
     0.066  [noun] લોકો કે વાહનોની કતાર જે કોઈની કે કશાકની રાહ જોઈ રહ્યા હોય
     0.055  [noun] એક પ્રકારની માળા જે ગળામાં પહેરવામાં આવે છે
     0.049  [noun] ગળામાં પહેરવાનું એક પ્રકારનું સોના, ચાંદી વગેરેનું ઘરેણું 

લગ્નમાં કન્યાએ સોનાનો સુંદર હાર પહેર્યો હતો   [target: હાર, method: lesk+emb]
     0.074  [noun] એક શેરની માળા
     0.075  [noun] ફૂલોની ગૂંથેલી માળા
     0.066  [noun] પરાજિત થવાની અવસ્થા કે ભાવ
     0.077  [noun] સુતર વગેરેમાં ગોળાકાર પરોવેલી કોઇ વસ્તુ જેમકે મણકા, ફૂલ આદિ જે ગળામાં પહેરવ
     0.066  [noun] એવી પરંપરા જેમાં એક જ પ્રકારની વસ્તુઓ, વ્યક્તિઓ કે જીવો એકબીજાની પાછળ એક કત


In [70]:
# WSD on the news corpus: real sentences containing the polysemous words (Step 2 sentence splitting)
MAX_HITS, MAX_SCAN = 60, 8000
targets = set(ambiguous)
occurrences = defaultdict(list)
for article in tqdm(df['text'].head(MAX_SCAN), desc='Collecting contexts'):
    for toks in sentence_tokens(article):
        for t in set(toks):
            l = lem(t)
            if l in targets and len(occurrences[l]) < MAX_HITS:
                occurrences[l].append(toks)
    if all(len(occurrences[l]) >= MAX_HITS for l in targets):
        break

corpus_preds = {}
for lemma in ambiguous:
    corpus_preds[lemma] = [disambiguate(lemma, remove_stopwords(toks), best_method)[0] for toks in occurrences[lemma]]
    counts = Counter(p.synset_id() for p in corpus_preds[lemma])
    print(f'{lemma}: {len(occurrences[lemma])} occurrences, {len(counts)} of {len(iwn.synsets(lemma))} senses used')
    for sid, n in counts.most_common(3):
        i = next(j for j, p in enumerate(corpus_preds[lemma]) if p.synset_id() == sid)
        s = corpus_preds[lemma][i]
        print(f'   {n:3d} x  [{s.pos()}] {s.gloss()[:70]}')
        print(f"          e.g. {' '.join(occurrences[lemma][i])[:110]}")
    print()

દિવસ: 60 occurrences, 8 of 8 senses used
    12 x  [noun] એ સમય જેમાં કોઈ વિશેષ વાત હોય
          e.g. આ કાર્યક્રમ માં આર એસ એસ હિન્દૂ સંગઠનના જનજાતિ સુરક્ષા મંચ ડાંગ નવસારીના સંયોજક વસંતભાઈ ગામીતે ધર્મતરણ થયેલ આદ
    11 x  [noun] ચોવીસ કલાકમાંથી એ સમય જે ઊંઘ પછી કામ કરવામાં પસાર થાય છે
          e.g. જેનાં કારણે ગાંધીનગર સિવિલનાં મેડિસિન વિભાગમાં સામાન્ય દિવસો કરતા ટકાથી વધુ દર્દીઓ સારવાર લેવા માટે આવી રહ્યા 
    11 x  [noun] હિન્દુ પંચાંગ પર આધારિત અઠવાડિયાનો કોઈ દિવસ
          e.g. પેજે ટ્વિટ કરીને જણાવ્યું છે કે જુલાઈ wwe સાથે મારો છેલ્લો દિવસ હશે

સમય: 60 occurrences, 8 of 9 senses used
    24 x  [noun] મિનિટો, કલાકો, વર્ષો વગેરેમાં મપાતું અંતર અથવા ગતિ જેનાથી ભૂત, વર્તમાન
          e.g. રાત્રી દરમિયાન પેટ્રોલિંગ સમયે પાદરડી ગામમાં પીએસઆઈની ટીમ પર હુમલો કરવામાં આવ્યો છે
     8 x  [noun] કોઈ એવો સમય કે પરિસ્થિતિ કે જેમાં કોઈ કાર્ય કે ઉદ્દેશ સહેલાઈથી અને જલ્
          e.g. કોવિડને લગતા તમામ લેટેસ્ટ અપડેટ અહીં વાંચો જેને શક્ય એટલા ઓછા સમયમાં પૂર્ણ કરવાનું રહેશે
     6 x  [noun] ભોજન 

In [71]:
# Contextual meaning without a dictionary (word sense induction): each occurrence of a word is represented
# by the averaged FastText vector of its context words, and the occurrences are clustered with k-means.
# Clusters with different typical context words point to different usages; the crosstab compares the
# clusters with the WordNet senses chosen by WSD.
from sklearn.cluster import KMeans

K = 3
target = max(ambiguous, key=lambda l: len(occurrences[l]))
contexts = [[t for t in remove_stopwords(toks) if lem(t) != target] for toks in occurrences[target]]
keep = [i for i, c in enumerate(contexts) if mean_vec('fasttext', c) is not None]
Xc = np.vstack([mean_vec('fasttext', contexts[i]) for i in keep])
km = KMeans(n_clusters=K, n_init=10, random_state=42).fit(Xc)

all_ctx = Counter(w for i in keep for w in contexts[i])
print(f'Target word: {target} ({len(keep)} occurrences)\n')
for c in range(K):
    words = Counter(w for i, lab in zip(keep, km.labels_) if lab == c for w in contexts[i])
    typical = sorted((w for w in words if all_ctx[w] >= 3), key=lambda w: (words[w] / all_ctx[w], words[w]), reverse=True)
    print(f'Cluster {c} ({(km.labels_ == c).sum()} occurrences) typical context words: {", ".join(typical[:12])}')

wsd_labels = [corpus_preds[target][i].gloss()[:45] for i in keep]
pd.crosstab(pd.Series(wsd_labels, name='WSD sense'), pd.Series(km.labels_, name='cluster'))

Target word: દિવસ (60 occurrences)

Cluster 0 (23 occurrences) typical context words: રહેશે, ડિસેમ્બર, રવિવાર, ભારે, આજથી, અનેક, ભાગોમાં, થવાની, વરસાદની, વરસાદ, હાઈ, બે
Cluster 1 (25 occurrences) typical context words: આજે, કોંગ્રેસ, આમ, આદમી, બૌધ્ધ, ત્રણ, વર્ષ, જશે, ગુજરાત, શકે, ગુજરાતના, થોડા
Cluster 2 (12 occurrences) typical context words: ન, બંધ, પડશે, માટેની, બીજા, દરમિયાન


cluster,0,1,2
WSD sense,,,
એ સમય જેમાં કોઈ વિશેષ વાત હોય,1,7,4
એક સૂર્યોદયથી બીજા સૂર્યોદય સુધીનો સમય જે ચોવ,1,3,3
કોઇ પણ ગ્રહને પોતાના અક્ષ પર એક વાર ફરવામાં લ,1,2,2
ચોવીસ કલાકમાંથી એ સમય જે ઊંઘ પછી કામ કરવામાં,4,4,3
નિશ્ચિત કે ઉચિત સમય,0,5,0
સારો સમય,1,0,0
સૂર્યોદયથી સૂર્યાસ્ત સુધીનો સમય,7,1,0
હિન્દુ પંચાંગ પર આધારિત અઠવાડિયાનો કોઈ દિવસ,8,3,0


### Step 9: Coreference Resolution

In [72]:
from itertools import count

PRONOUNS = {'તે', 'તેણે', 'તેને', 'તેના', 'તેની', 'તેનું', 'તેનો', 'તેઓ', 'તેઓએ', 'તેઓને', 'તેઓના',
            'તેમણે', 'તેમને', 'તેમના', 'તેમની', 'તેમનું', 'તેમનો', 'એ', 'એણે', 'એને', 'એના', 'એની', 'એનું'}
PLURAL_PRONOUNS = {'તેઓ', 'તેઓએ', 'તેઓને', 'તેઓના'}
DETERMINER_LIKE = {'તે', 'એ'}     # "તે સમયે" = "at that time": a determiner, not a pronoun, before a noun
CASE_SUFFIXES = sorted(['માંથી', 'માં', 'નાં', 'ના', 'ની', 'નું', 'નો', 'ને', 'થી', 'એ', 'ે'], key=len, reverse=True)


def base_form(w):
    for s in CASE_SUFFIXES:
        if w.endswith(s) and len(w) - len(s) >= 2:
            return w[:-len(s)]
    return w


def tag_tokens(toks):
    tags = [t for _, t in hmm_tagger.tag(toks)]
    fixed = list(tags)
    for i, (w, t) in enumerate(zip(toks, tags)):
        nxt = tags[i + 1] if i + 1 < len(tags) else 'PUNCT'
        if t == 'VERB' and w.endswith('ે') and nxt not in ('VERB', 'PUNCT', 'CCONJ'):
            fixed[i] = 'NOUN'
    return fixed


def find_mentions(sentences):
    mentions = []
    for si, toks in enumerate(sentences):
        tags = tag_tokens(toks)
        i = 0
        while i < len(toks):
            w = toks[i]
            if w in PRONOUNS:
                if not (w in DETERMINER_LIKE and i + 1 < len(toks) and tags[i + 1] in ('NOUN', 'PROPN')):
                    mentions.append({'sent': si, 'pos': i, 'len': 1, 'text': w, 'kind': 'pronoun',
                                     'plural': w in PLURAL_PRONOUNS})
                i += 1
            elif tags[i] in ('NOUN', 'PROPN'):
                j = i
                while j + 1 < len(toks) and tags[j + 1] in ('NOUN', 'PROPN') and toks[j + 1] not in PRONOUNS:
                    j += 1
                head = toks[j]
                mentions.append({'sent': si, 'pos': i, 'len': j - i + 1, 'text': ' '.join(toks[i:j + 1]),
                                 'kind': 'noun', 'key': base_form(head), 'subject': head.endswith(('એ', 'ે')),
                                 'plural': base_form(head).endswith(('ો', 'ઓ'))})
                i = j + 1
            else:
                i += 1
    return mentions


def coref(sentences, max_back=2):
    """Returns mentions, entity id of every mention, and (pronoun, antecedent) mention-index links."""
    mentions = find_mentions(sentences)
    ids, key_entity, entity, links = count(), {}, {}, []
    for k, m in enumerate(mentions):
        if m['kind'] == 'noun':
            if m['key'] not in key_entity:
                key_entity[m['key']] = next(ids)
            entity[k] = key_entity[m['key']]
            continue
        cands = [c for c in range(k) if mentions[c]['kind'] == 'noun' and m['sent'] - mentions[c]['sent'] <= max_back]
        if not cands:
            entity[k] = next(ids)
            continue
        best = max(cands, key=lambda c: (mentions[c]['subject'], mentions[c]['plural'] == m['plural'], c))
        entity[k] = entity[best]
        links.append((k, best))
    return mentions, entity, links


def annotate(sentences, mentions, entity):
    """Print the text with every mention of a multi-mention entity marked as [mention]entity-id."""
    sizes = Counter(entity.values())
    starts = {(m['sent'], m['pos']): k for k, m in enumerate(mentions)}
    for si, toks in enumerate(sentences):
        out, i = [], 0
        while i < len(toks):
            k = starts.get((si, i))
            if k is None:
                out.append(toks[i])
                i += 1
            else:
                m = mentions[k]
                out.append(f"[{m['text']}]{entity[k]}" if sizes[entity[k]] > 1 else m['text'])
                i += m['len']
        print(f's{si + 1}: ' + ' '.join(out))


def show_chains(mentions, entity, min_size=2):
    chains = defaultdict(list)
    for k, m in enumerate(mentions):
        chains[entity[k]].append(m)
    for e, ms in sorted(chains.items(), key=lambda x: -len(x[1])):
        if len(ms) >= min_size:
            print(f'Entity {e}: ' + '  →  '.join(f"{m['text']} (s{m['sent'] + 1})" for m in ms))

In [ ]:
# Demo on a short news-style paragraph, with the expected antecedent of each pronoun
SAMPLE = ('મુખ્યમંત્રી ભૂપેન્દ્ર પટેલે આજે ગાંધીનગરમાં નવી યોજનાની જાહેરાત કરી. '
          'તેમણે કહ્યું કે આ યોજનાથી ખેડૂતોને સીધો લાભ મળશે. '
          'ખેડૂતોએ યોજનાનું સ્વાગત કર્યું, પરંતુ તેઓ પાક વીમાની રકમ અંગે ચિંતિત છે. '
          'વિપક્ષના નેતાએ યોજનાની ટીકા કરી. '
          'તેણે કહ્યું કે સરકારે અગાઉનાં વચનો પૂરાં કર્યાં નથી.')
EXPECTED = {'તેમણે': 'પટેલ', 'તેઓ': 'ખેડૂતો', 'તેણે': 'નેતા'}

sample_sents = list(pos_sentence_tokens(SAMPLE))
mentions, entity, links = coref(sample_sents)
annotate(sample_sents, mentions, entity)
print('\nCoreference chains:')
show_chains(mentions, entity)
print('\nPronoun resolution:')
for p, a in links:
    pron, ante = mentions[p], mentions[a]
    ok = EXPECTED.get(pron['text']) == ante['key']
    print(f"  {pron['text']} (s{pron['sent'] + 1})  →  {ante['text']} (s{ante['sent'] + 1})   "
          f"{'✓' if ok else 'X expected ' + EXPECTED.get(pron['text'], '?')}")

s1: [મુખ્યમંત્રી ભૂપેન્દ્ર પટેલે]0 આજે ગાંધીનગરમાં નવી [યોજનાની]2 જાહેરાત કરી .
s2: [તેમણે]0 કહ્યું કે આ યોજનાથી ખેડૂતોને સીધો લાભ મળશે .
s3: [ખેડૂતોએ યોજનાનું]2 સ્વાગત કર્યું , પરંતુ [તેઓ]0 પાક વીમાની રકમ અંગે ચિંતિત છે .
s4: [વિપક્ષના નેતાએ યોજનાની ટીકા]6 કરી .
s5: [તેણે]6 કહ્યું કે સરકારે અગાઉનાં વચનો પૂરાં કર્યાં નથી .

Coreference chains:
Entity 0: મુખ્યમંત્રી ભૂપેન્દ્ર પટેલે (s1)  →  તેમણે (s2)  →  તેઓ (s3)
Entity 2: યોજનાની (s1)  →  ખેડૂતોએ યોજનાનું (s3)
Entity 6: વિપક્ષના નેતાએ યોજનાની ટીકા (s4)  →  તેણે (s5)

Pronoun resolution:
  તેમણે (s2)  →  મુખ્યમંત્રી ભૂપેન્દ્ર પટેલે (s1)   ✓
  તેઓ (s3)  →  મુખ્યમંત્રી ભૂપેન્દ્ર પટેલે (s1)   ✗ expected ખેડૂતો
  તેણે (s5)  →  વિપક્ષના નેતાએ યોજનાની ટીકા (s4)   ✗ expected નેતા


In [74]:
# Coreference on corpus articles: statistics, plus one annotated article
N_COREF_ARTICLES = 500
stats, continuity, demo = Counter(), [], None
for idx, article in enumerate(tqdm(df['text'].head(N_COREF_ARTICLES), desc='Coreference')):
    sents = list(pos_sentence_tokens(article))
    mentions, entity, links = coref(sents)
    sizes = Counter(entity.values())
    stats['articles'] += 1
    stats['sentences'] += len(sents)
    stats['mentions'] += len(mentions)
    stats['pronouns'] += sum(m['kind'] == 'pronoun' for m in mentions)
    stats['pronouns resolved'] += len(links)
    stats['entities with 2+ mentions'] += sum(n >= 2 for n in sizes.values())
    # entity continuity: does a sentence mention an entity (with 2+ mentions) from the previous sentence?
    per_sent = defaultdict(set)
    for k, m in enumerate(mentions):
        if sizes[entity[k]] > 1:
            per_sent[m['sent']].add(entity[k])
    continuity += [bool(per_sent[s] & per_sent[s - 1]) for s in range(1, len(sents))]
    if demo is None and len(links) >= 2 and len(sents) <= 10:
        demo = (idx, sents, mentions, entity)

print(pd.Series(stats).to_string())
print(f"Pronouns resolved: {stats['pronouns resolved'] / max(stats['pronouns'], 1):.1%}")

if demo:
    idx, sents, mentions, entity = demo
    print(f'\nArticle {idx}: {headlines[idx]}\n')
    annotate(sents, mentions, entity)
    print('\nCoreference chains:')
    show_chains(mentions, entity)

Coreference: 100%|██████████| 500/500 [00:02<00:00, 233.29it/s]

articles                       500
sentences                     8823
mentions                     37346
pronouns                      2345
pronouns resolved             2345
entities with 2+ mentions     6349
Pronouns resolved: 100.0%

Article 11: ડાંગ જિલ્લામાં જનજાતિ સુરક્ષા મંચ દ્વારા ડી-લિસ્ટિંગ મુદ્દે વિશાળ રેલી જાહેરસભા મા આદિવાસીઓ ની જનમેદની ઉમટી   આદિવાસીઓને મળતા હક્કો ધર્માંતરણ કરેલ પરિવારો ને ન આપવાની ઘોષણા કરી : વસંતભાઈ ગામીત

s1: દેશમાં ધાર્મિક ઉત્સવ દરમિયાન હિંસા ના [બનાવો]1 સામે આવી રહ્યા છે આવા સમયે ડાંગમાં જનજાતિ સુરક્ષા મંચ દ્વારા ડી - [લિસ્ટિંગ ની માંગ]4 સાથે રેલી નું આયોજન કરવામાં આવ્યું [હતું]6 .
s2: ડાંગ ના મુખ્ય મથક આહવામાં આદિવાસી સમાજ દ્વારા હાથમાં પ્લેકાર્ડ સાથે જય શ્રી રામના નારા સાથે વિશાળ રેલીમાં લોકો રંગઉપવન ખાતે જાહેર સભામાં હજારો ની સંખ્યા મા આદિવાસીઓ ઉમટી પડયા હતા .
s3: આ કાર્યક્રમ માં આર . એસ . એસ હિન્દૂ સંગઠનના જનજાતિ સુરક્ષા મંચ ડાંગ નવસારીના સંયોજક વસંતભાઈ ગામીતે ધર્મતરણ થયેલ આદિવાસીઓને સનાતન ધર્મમાં ઘર વાપસી કરવા આહવાન કર્યું [હતું]6 , તેમજ [તેઓ]1 

In [ ]:
# DISCOURSE STRUCTURE
CONNECTIVES = {
    'અને': 'Addition', 'તેમજ': 'Addition', 'ઉપરાંત': 'Addition', 'આ ઉપરાંત': 'Addition', 'વળી': 'Addition',
    'પરંતુ': 'Contrast', 'જોકે': 'Contrast', 'છતાં': 'Contrast', 'તેમ છતાં': 'Contrast',
    'કારણ કે': 'Cause', 'કેમકે': 'Cause', 'કારણકે': 'Cause',
    'તેથી': 'Result', 'એટલે': 'Result', 'જેથી': 'Result', 'તેના કારણે': 'Result', 'જેના કારણે': 'Result',
    'ત્યારે': 'Temporal', 'ત્યારબાદ': 'Temporal', 'બાદમાં': 'Temporal', 'તે પછી': 'Temporal', 'જ્યારે': 'Temporal',
    'જો': 'Condition',
}
N_DISCOURSE_ARTICLES = 2000

rel_counts, examples, n_sents = Counter(), {}, 0
for article in tqdm(df['text'].head(N_DISCOURSE_ARTICLES), desc='Connectives'):
    for toks in pos_sentence_tokens(article):
        words = [t for t in toks if unicodedata.category(t[0])[0] != 'P']
        n_sents += 1
        i = 0
        while i < len(words):
            two = ' '.join(words[i:i + 2])
            conn = two if two in CONNECTIVES else words[i] if words[i] in CONNECTIVES else None
            if conn:
                position = 'sentence-initial (links to previous sentence)' if i == 0 else 'inside sentence'
                rel_counts[(CONNECTIVES[conn], position)] += 1
                examples.setdefault(CONNECTIVES[conn], ' '.join(toks)[:120])
                i += len(conn.split())
            else:
                i += 1

print(f'Sentences analysed: {n_sents}')
print(f'Entity continuity between adjacent sentences: {np.mean(continuity):.1%} continue, '
      f'{1 - np.mean(continuity):.1%} shift (from {len(continuity)} sentence pairs)\n')
display(pd.Series(rel_counts).unstack(fill_value=0).assign(total=lambda d: d.sum(axis=1)).sort_values('total', ascending=False))
print('Example sentence for each relation:')
for rel, ex in examples.items():
    print(f'  {rel:10s} {ex}')

Connectives: 100%|██████████| 2000/2000 [00:01<00:00, 1037.77it/s]

Sentences analysed: 35941
Entity continuity between adjacent sentences: 25.0% continue, 75.0% shift (from 8323 sentence pairs)



,inside sentence,sentence-initial (links to previous sentence),total
Addition,11128,501,11629
Temporal,1595,919,2514
Contrast,946,725,1671
Result,650,426,1076
Condition,446,568,1014
Cause,164,80,244


Example sentence for each relation:
  Addition   7માં લાડી લોહાણા સિંધી પંચાયત બાંટવા દ્વારા આયોજિત લાડી લોહાણા સિંધી નવયુવક મંડળ બાંટવા તેમજ શ્રી લાડી લોહાણા મહિલા મંડળ
  Result     જેના કારણે કાદવ કીચડ તેમજ ગંદકીનું સામ્રાજ્ય ઉભું થઈ ચૂક્યું છે .
  Contrast   પરંતુ હાલ WWEના ચાહકો માટે નિરાશાજનક ખબર સામે આવ્યા છે .
  Temporal   પેજ 7 જુલાઈએ કરાર સમાપ્ત થશે ત્યારે WWE છોડી દેશે .
  Cause      અમેરિકાના ડલ્લાસ વાઈઝને પણ સિલ્વર એનાયત કરવામાં આવ્યો હતો કારણ કે તેણે અને કુમાર , બન્ને એ 2.06 મીટર ઊંચો કૂદકો લગાવ્યો 
  Condition  પૂણે પાસે આવેલોછે આ નાનેઘાટ હવા વધુ હોવાથી આ ઘોઘલ જાણે ઉપર સાઈડ વહેતો દેખાઈ છે સોમાસુ આવતાની સાથએ જ કુદરત સોળે કળાએ ખીલી
